# Spin Monte Carlo on the generated multi-colour kernel — ready to run

Upload this file to Kaggle or Colab, or open it in Jupyter on your own machine, and press **Run all**.
**You do not need to know Python, GPUs or the repository**: the notebook carries the package modules it
uses, each checked against its `sha256`, so it runs code byte-identical to `spin-colour-kernel` at
commit `f2e9bb5`.

## What it does

It cools every *state* you define (one set θ of Hamiltonian parameters) from `T_MAX` down to `T_MIN`,
with many independent replicas per state, and records at every temperature the energy, the
magnetization vector, |M| and M² (so the specific heat Cv and the susceptibility χ follow), and the full
spin configuration of every replica.

## How to use it

1. **Edit only cell 1 (the parameters).** Every setting is explained next to it.
2. *Run all*. The notebook finds your hardware by itself: every NVIDIA GPU it sees (one process per GPU,
   states dealt round-robin), or the CPU when there is none.
3. Read the plots (E, |M|, Cv, χ against T) and download the output folder.

> GPU: an NVIDIA GPU with compute capability ≥ 7.0 (T4, A100, H100, …; Triton does not run on a P100).
> CPU: Linux only (Colab, Kaggle without accelerator, WSL), through the Triton interpreter. It is
> correct but **hundreds of times slower**: keep the box, the replicas and the sweeps small there.

## 1 · Parameters

The only cell to edit.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════════════════════
#  1 · THE PARAMETERS — the only cell to edit
# ══════════════════════════════════════════════════════════════════════════════════════════════
#
# ── 1.1  TYPE OF HAMILTONIAN ────────────────────────────────────────────────────────────────────
#   "ising"       spins s = ±1                   (1 component)
#   "xy"          unit spins in the plane         (2 components)
#   "heisenberg"  unit spins in 3D                (3 components)
MODEL = "heisenberg"

# ── 1.2  THE HAMILTONIAN (how it is written) ────────────────────────────────────────────────────
#
#   H = − Σ_s J_s Σ_<ij> in shell s  S_i · S_j    exchange, one constant per neighbour shell (each pair once)
#       − Σ_<ij>  S_i^T A_ij S_j      interfacial DMI on in-plane nearest-neighbour bonds, A_lm = D Σ_k r̂_k ε_klm
#       + Σ_i K1_i (sx² sy² + sx² sz² + sy² sz²)  +  K2 Σ_i (sx sy sz)²      cubic anisotropy
#       − Σ_i  H · S_i                                           Zeeman field
#
#   J_s > 0 is ferromagnetic, J_s < 0 antiferromagnetic. Shell 1 = nearest neighbours, 2 = next
#   nearest, and so on (shells 1–4 on sc). K1_i = Ks on SURFACE sites (an active site with fewer active
#   nearest neighbours than a bulk site), K1 elsewhere.
#   Field direction: Heisenberg (sinθ cosφ, sinθ sinφ, cosθ) with θ = H_theta, φ = H_phi in degrees
#   (θ = 0 is +z, the normal of a film or disc); XY (cosφ, sinφ); Ising +1.
#   D, K1, Ks and K2 exist only for "heisenberg"; Ising and XY carry J and H.
#
# ── 1.3  THE STATES: one set θ of parameters each — add as many as you want ─────────────────────
#   name     a label for the plots and files
#   J        {shell: constant}, e.g. {1: 1.0} or {1: 1.0, 2: -0.2}
#   D        interfacial Dzyaloshinskii–Moriya constant
#   K1, Ks   cubic anisotropy, bulk and surface (Ks left out = K1)
#   K2       second-order cubic anisotropy
#   H, H_theta, H_phi   field magnitude and direction (degrees)
#   A parameter left out is zero.
STATES = [
    dict(name="ferromagnet", J={1: 1.0}),
    dict(name="with field",  J={1: 1.0}, H=0.2, H_theta=0.0),
    # dict(name="nanodisk", J={1: 1.0}, D=0.88, K1=0.177, Ks=0.139, K2=0.034, H=0.483, H_theta=0.0),
]

# ── 1.4  UNITS ──────────────────────────────────────────────────────────────────────────────────
#   "reduced"  energies in units of your J, k_B = 1 (T is in the same units)
#   "meV"      energies in meV, T in kelvin (k_B = 0.086173404 meV/K)
UNITS = "reduced"

# ── 1.5  CRYSTAL STRUCTURE AND BOX ──────────────────────────────────────────────────────────────
#   2D: "square", "triangular", "honeycomb", "kagome"      BOX = [Lx, Ly]
#   3D: "sc", "bcc", "fcc" (bcc, fcc: conventional cubic cells of 2 and 4 sites)   BOX = [Lx, Ly, Lz]
#   The box is counted in unit cells; a square box has equal sizes, a rectangular one does not.
#   PERIODIC: one True/False per axis. If no colouring fits a periodic box, it falls back to open.
CRYSTAL  = "sc"
BOX      = [12, 12, 12]
PERIODIC = [True, True, True]

# ── 1.6  ACTIVE SITES (a shape cut out of the box) ──────────────────────────────────────────────
#   First the coordinate system, then the shape written in it. The origin is the centre of the box;
#   lengths are in lattice units; angles in radians.
#     "cartesian"    x, y, z
#     "spherical"    r, theta, phi        a sphere of radius 5:            SHAPE = "r <= 5"
#     "cylindrical"  rho, phi, z          a disc of radius 5, 3 layers:    SHAPE = "(rho <= 5) & (abs(z) <= 1)"
#   Combine conditions with & (and), | (or), ~ (not). Leave BOTH empty to keep every site active.
#   Sites outside the shape carry no spin; a shape usually goes with PERIODIC = [False, False, False].
COORDS = ""
SHAPE  = ""

# ── 1.7  TEMPERATURES AND SWEEPS ────────────────────────────────────────────────────────────────
T_MAX  = 2.0          # first (hottest) temperature
T_MIN  = 0.2          # last (coldest) temperature
T_STEP = 0.1          # step of the ladder
THERMALIZATION = 1000 # sweeps at each temperature before measuring
MEASUREMENT    = 2000 # sweeps at each temperature while measuring
MEASURE_EVERY  = 10   # read E and M every this many measuring sweeps
REPLICAS       = 32   # independent chains per state (more is cheaper per spin update on a GPU)
SEED           = 1

# ── 1.8  OUTPUT ─────────────────────────────────────────────────────────────────────────────────
#   CONFIG_EVERY = 1 saves every replica's full configuration at every temperature step;
#   2 at every second step, and so on.
CONFIG_EVERY = 1
OUTPUT_DIR   = "simulation_output"
GPUS         = "all"   # "all", or a list such as [0] or [0, 1]; ignored on CPU

## 2 · Hardware

In [ ]:
# 2 · Hardware: every usable NVIDIA GPU, or the CPU through the Triton interpreter.
import os, subprocess, sys

def _sh(*a):
    try:
        return subprocess.run(a, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""

import torch
caps = [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())] \
    if torch.cuda.is_available() else []
usable = [i for i, c in enumerate(caps) if c >= (7, 0)]
if caps and not usable:
    print(f"GPU(s) found with capability {caps}, below the 7.0 Triton needs: using the CPU")
if GPUS != "all":
    usable = [g for g in usable if g in list(GPUS)]
DEVICE = "cuda" if usable else "cpu"
if DEVICE == "cpu":
    os.environ["TRITON_INTERPRET"] = "1"      # must precede the first import of triton
try:
    import triton
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton
if DEVICE == "cuda":
    print("GPU:", _sh("nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"))
    print(f"using {len(usable)} GPU(s): {usable}")
else:
    print("no usable GPU: running on the CPU (Triton interpreter, slow; keep the run small)")
print(f"torch {torch.__version__}  triton {triton.__version__}")

## 3 · The modules

Written to disk from the literals below, each checked against its `sha256`.

In [ ]:
# Spin_Colour_Kernel/__init__.py — byte-identical to the repository. sha256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
import hashlib, os
_src = ''
assert hashlib.sha256(_src.encode()).hexdigest() == "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "__init__.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/__init__.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/__init__.py  sha256 e3b0c44298fc1c14…")

In [ ]:
# Spin_Colour_Kernel/crystals.py — byte-identical to the repository. sha256 13b8e7758dcc9259ee1d7705e6b994857a7a0c80f72e73d6998195ad562dfe76
import hashlib, os
_src = '"""Crystals as Bravais lattices with a basis, and their bond tables (proposal r07, eq. 4).\n\nA site is the pair (c, alpha): an integer cell coordinate c in Z^D and a basis index alpha.\nA directed bond (alpha, alpha2, delta) joins (c, alpha) to (c + delta, alpha2). Bond tables are\ngenerated from neighbour shells by real-space distance, so a crystal is fully described by its\nprimitive (or conventional) vectors and its basis offsets.\n"""\n\nfrom __future__ import annotations\n\nimport itertools\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Bond:\n    """One directed bond-table entry: (c, alpha) ~ (c + delta, alpha2), on neighbour shell `shell`."""\n\n    alpha: int\n    alpha2: int\n    delta: tuple[int, ...]\n    shell: int\n\n\n@dataclass\nclass Crystal:\n    name: str\n    vectors: np.ndarray          # (D, D) rows are lattice vectors a_k in Cartesian units\n    basis: np.ndarray            # (b, D) basis offsets in fractional (cell) coordinates\n    shells: tuple[int, ...] = (1,)\n    bonds: list[Bond] = field(default_factory=list)\n\n    @property\n    def D(self) -> int:\n        return int(self.vectors.shape[0])\n\n    @property\n    def b(self) -> int:\n        return int(self.basis.shape[0])\n\n    def position(self, c, alpha) -> np.ndarray:\n        frac = np.asarray(c, dtype=float) + self.basis[alpha]\n        return frac @ self.vectors\n\n\ndef shell_distances(vectors: np.ndarray, basis: np.ndarray, nshell: int, reach: int = 3) -> list[float]:\n    """The first `nshell` distinct neighbour distances over every basis pair."""\n    D = vectors.shape[0]\n    ds = set()\n    for a in range(basis.shape[0]):\n        for a2 in range(basis.shape[0]):\n            for delta in itertools.product(range(-reach, reach + 1), repeat=D):\n                r = (np.asarray(delta, float) + basis[a2] - basis[a]) @ vectors\n                d = float(np.linalg.norm(r))\n                if d > 1e-9:\n                    ds.add(round(d, 9))\n    return sorted(ds)[:nshell]\n\n\ndef build_bonds(vectors: np.ndarray, basis: np.ndarray, shells: tuple[int, ...], reach: int = 3) -> list[Bond]:\n    """Directed bond table for the requested shells (1-based), in a fixed, documented order.\n\n    Order: shell, then alpha, then alpha2, then delta in lexicographic order. The order is the\n    accumulation order of the local field, both in the kernel and in every reference.\n    """\n    dist = shell_distances(vectors, basis, max(shells), reach)\n    D = vectors.shape[0]\n    out: list[Bond] = []\n    for s in sorted(shells):\n        target = dist[s - 1]\n        for a in range(basis.shape[0]):\n            for a2 in range(basis.shape[0]):\n                for delta in itertools.product(range(-reach, reach + 1), repeat=D):\n                    r = (np.asarray(delta, float) + basis[a2] - basis[a]) @ vectors\n                    if abs(np.linalg.norm(r) - target) < 1e-7:\n                        out.append(Bond(a, a2, tuple(int(v) for v in delta), s))\n    return out\n\n\ndef make(name: str, shells: tuple[int, ...] = (1,)) -> Crystal:\n    """The crystals of the system matrix. bcc and fcc use conventional cubic cells (b = 2, 4)."""\n    s3 = np.sqrt(3.0)\n    tri = np.array([[1.0, 0.0], [0.5, s3 / 2]])\n    table = {\n        "square": (np.eye(2), np.zeros((1, 2))),\n        "triangular": (tri, np.zeros((1, 2))),\n        "honeycomb": (tri, np.array([[0.0, 0.0], [1 / 3, 1 / 3]])),\n        "kagome": (tri, np.array([[0.0, 0.0], [0.5, 0.0], [0.0, 0.5]])),\n        "sc": (np.eye(3), np.zeros((1, 3))),\n        "bcc": (np.eye(3), np.array([[0, 0, 0], [0.5, 0.5, 0.5]])),\n        "fcc": (np.eye(3), np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]])),\n    }\n    if name not in table:\n        raise KeyError(f"unknown crystal {name!r}; known: {sorted(table)}")\n    vec, basis = table[name]\n    vec = np.asarray(vec, float)\n    basis = np.asarray(basis, float)\n    reach = 3 if max(shells) <= 2 else 4\n    return Crystal(name, vec, basis, tuple(shells), build_bonds(vec, basis, tuple(shells), reach))\n\n\ndef coordination(crystal: Crystal) -> dict[int, int]:\n    """Bonds leaving each basis site, per alpha."""\n    out = {a: 0 for a in range(crystal.b)}\n    for bd in crystal.bonds:\n        out[bd.alpha] += 1\n    return out\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "13b8e7758dcc9259ee1d7705e6b994857a7a0c80f72e73d6998195ad562dfe76", "crystals.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/crystals.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/crystals.py  sha256 13b8e7758dcc9259…")

In [ ]:
# Spin_Colour_Kernel/colouring.py — byte-identical to the repository. sha256 86c87e437035d5af8e4ead9416924b598beb80241722d45742de20fa66d26a01
import hashlib, os
_src = '"""Linear colourings as lattice homomorphisms into a finite abelian group (proposal r07, eqs. 5-8).\n\ngamma(c, alpha) = K c + o_alpha (mod m), with K in Z^{r x D} and o_alpha in Z_m1 x ... x Z_mr.\nAdmissibility (7) is a finite check over the bond table; the seam condition (8) binds the colouring\nto a periodic box. The search runs over every abelian group (invariant-factor form) in increasing\norder n_c and returns the first admissible colouring. The clique number of the bond graph is the\nlower bound, so optimality is certified when the two meet.\n"""\n\nfrom __future__ import annotations\n\nimport itertools\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .crystals import Crystal\n\n\n@dataclass(frozen=True)\nclass Colouring:\n    m: tuple[int, ...]                 # group orders, invariant-factor form\n    K: tuple[tuple[int, ...], ...]     # r rows, D columns\n    o: tuple[tuple[int, ...], ...]     # b rows, r columns\n\n    @property\n    def n_colours(self) -> int:\n        return int(np.prod(self.m)) if self.m else 1\n\n    def colour(self, c, alpha) -> int:\n        """Mixed-radix index of gamma(c, alpha) in [0, n_c)."""\n        idx = 0\n        for r, mr in enumerate(self.m):\n            g = (sum(self.K[r][k] * int(c[k]) for k in range(len(c))) + self.o[alpha][r]) % mr\n            idx = idx * mr + g\n        return idx\n\n    def describe(self) -> str:\n        grp = " x ".join(f"Z{m}" for m in self.m) or "trivial"\n        return f"{grp}; K={[list(r) for r in self.K]}; o={[list(x) for x in self.o]}"\n\n\ndef abelian_groups(n: int) -> list[tuple[int, ...]]:\n    """Every abelian group of order n, as invariant factors m1 | m2 | ... | mr."""\n    out = []\n\n    def rec(rem, prev, acc):\n        if rem == 1:\n            out.append(tuple(acc))\n            return\n        for d in range(2, rem + 1):\n            if rem % d == 0 and (prev is None or d % prev == 0):\n                rec(rem // d, d, acc + [d])\n\n    if n == 1:\n        return [()]\n    rec(n, None, [])\n    # invariant factors must divide each other in order; rec builds m1 | m2 | ...\n    return [g for g in out if all(g[i + 1] % g[i] == 0 for i in range(len(g) - 1))]\n\n\ndef admissible(col: Colouring, crystal: Crystal) -> bool:\n    """Condition (7): K delta + o_alpha2 - o_alpha != 0 (mod m) for every bond."""\n    for bd in crystal.bonds:\n        v = []\n        for r, mr in enumerate(col.m):\n            v.append((sum(col.K[r][k] * bd.delta[k] for k in range(crystal.D))\n                      + col.o[bd.alpha2][r] - col.o[bd.alpha][r]) % mr)\n        if all(x == 0 for x in v):\n            return False\n    return True\n\n\ndef seam_ok(col: Colouring, box: tuple[int, ...], periodic: tuple[bool, ...]) -> bool:\n    """Condition (8): every periodic axis k has K[:, k] L_k = 0 (mod m)."""\n    for k, (Lk, per) in enumerate(zip(box, periodic)):\n        if per and any((col.K[r][k] * Lk) % mr for r, mr in enumerate(col.m)):\n            return False\n    return True\n\n\ndef search(crystal: Crystal, box: tuple[int, ...] | None = None, periodic: tuple[bool, ...] | None = None,\n           max_colours: int = 32) -> Colouring | None:\n    """First admissible linear colouring in increasing n_c (and seam-compatible with `box` if given)."""\n    D, b = crystal.D, crystal.b\n    for n in range(1, max_colours + 1):\n        for m in abelian_groups(n):\n            r = len(m)\n            elems = list(itertools.product(*[range(mi) for mi in m])) if r else [()]\n            # K columns: one group element per lattice axis\n            for cols in itertools.product(elems, repeat=D):\n                K = tuple(tuple(cols[k][i] for k in range(D)) for i in range(r))\n                for offs in itertools.product(elems, repeat=b - 1):\n                    o = (tuple([0] * r),) + tuple(offs)\n                    col = Colouring(m, K, o)\n                    if box is not None and not seam_ok(col, box, periodic or (True,) * D):\n                        continue\n                    if admissible(col, crystal):\n                        return col\n    return None\n\n\ndef verify_on_box(col: Colouring, crystal: Crystal, box: tuple[int, ...], periodic: tuple[bool, ...]) -> bool:\n    """Independent check: colour every site of the box and test every bond explicitly, seam included."""\n    for c in itertools.product(*[range(L) for L in box]):\n        for bd in crystal.bonds:\n            if bd.alpha >= crystal.b:\n                continue\n            c2 = []\n            inside = True\n            for k in range(crystal.D):\n                x = c[k] + bd.delta[k]\n                if periodic[k]:\n                    x %= box[k]\n                elif not 0 <= x < box[k]:\n                    inside = False\n                c2.append(x)\n            if inside and col.colour(c, bd.alpha) == col.colour(c2, bd.alpha2):\n                return False\n    return True\n\n\ndef clique_number(crystal: Crystal, window: int = 2) -> int:\n    """Largest clique of the infinite bond graph, by Bron-Kerbosch on a window around one site.\n\n    Every clique is translated so that one of its sites is (0, alpha); every other site lies within\n    one bond of it, so a window of the bond reach holds every clique.\n    """\n    D = crystal.D\n    reach = max(max(abs(x) for x in bd.delta) for bd in crystal.bonds)\n    w = max(window, reach)\n    nodes = [(c, a) for c in itertools.product(range(-w, w + 1), repeat=D) for a in range(crystal.b)]\n    index = {n: i for i, n in enumerate(nodes)}\n\n    nbrs = [set() for _ in nodes]\n    for i, u in enumerate(nodes):\n        for bd in crystal.bonds:\n            if bd.alpha != u[1]:\n                continue\n            c2 = tuple(u[0][k] + bd.delta[k] for k in range(D))\n            j = index.get((c2, bd.alpha2))\n            if j is not None:\n                nbrs[i].add(j)\n    best = 1\n    for a in range(crystal.b):\n        start = index[(tuple([0] * D), a)]\n\n        def bk(R, P, X):\n            nonlocal best\n            if not P and not X:\n                best = max(best, len(R))\n                return\n            if len(R) + len(P) <= best:\n                return\n            for v in list(P):\n                bk(R | {v}, P & nbrs[v], X & nbrs[v])\n                P = P - {v}\n                X = X | {v}\n\n        bk({start}, set(nbrs[start]), set())\n    return best\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "86c87e437035d5af8e4ead9416924b598beb80241722d45742de20fa66d26a01", "colouring.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/colouring.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/colouring.py  sha256 86c87e437035d5af…")

In [ ]:
# Spin_Colour_Kernel/system.py — byte-identical to the repository. sha256 f3755bd076a9c0cd28a3da7dc7c774d2db5188f5e84085dce981f276c33fde03
import hashlib, os
_src = '"""A system is the triple (spin model, crystal, finite geometry) with its couplings (proposal r07, eqs. 1-4).\n\nSpins live in the component-planar, basis-planar layout of eq. (10):\n    S[B, n, b, L3, L2, L1], fastest axis L1 (x). For D = 2, L3 = 1.\nMasked sites hold the zero vector and are never updated, so a masked or open-boundary neighbour\ncontributes zero to the local field (3) without a mask load.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom . import crystals as cr\nfrom .colouring import Colouring, search, seam_ok, verify_on_box\n\n\n@dataclass\nclass System:\n    name: str\n    n: int                                   # spin components: 1 Ising, 2 XY, 3 Heisenberg\n    crystal: cr.Crystal\n    box: tuple[int, ...]                     # (L1, L2[, L3]) cells\n    periodic: tuple[bool, ...]\n    J: dict[int, float]                      # isotropic exchange per shell\n    extra: dict[int, np.ndarray] = field(default_factory=dict)   # bond index -> extra n x n matrix (DMI ...)\n    h: tuple[float, ...] | None = None       # uniform field\n    cubic_k1: np.ndarray | None = None       # per-site cubic anisotropy K1, shape (b, L3, L2, L1)\n    cubic_k2: float = 0.0                    # (sx sy sz)^2 coefficient\n    mask: np.ndarray | None = None           # bool (b, L3, L2, L1); False = no spin\n    colouring: Colouring | None = None\n    T_ref: float | None = None               # reference critical temperature, when one exists\n    acceptance: str = "metropolis"           # "metropolis": u < exp(-beta dE); "barker": heat-bath rule\n\n    def __post_init__(self):\n        if self.n not in (1, 2, 3):\n            raise ValueError("n must be 1, 2 or 3")\n        if len(self.box) != self.crystal.D or len(self.periodic) != self.crystal.D:\n            raise ValueError("box and periodic must have one entry per lattice axis")\n        if self.cubic_k1 is not None and self.n != 3:\n            raise ValueError("cubic anisotropy needs n = 3")\n        if self.acceptance not in ("metropolis", "barker"):\n            raise ValueError("acceptance must be \'metropolis\' or \'barker\'")\n        if self.colouring is None:\n            self.colouring = search(self.crystal, self.box, self.periodic)\n            if self.colouring is None:\n                raise ValueError(f"{self.name}: no seam-compatible linear colouring for box {self.box}")\n        if not seam_ok(self.colouring, self.box, self.periodic):\n            raise ValueError(f"{self.name}: colouring violates the seam condition (8) on box {self.box}")\n        if not verify_on_box(self.colouring, self.crystal, self.box, self.periodic):\n            raise ValueError(f"{self.name}: colouring is not admissible on the box")\n\n    # ---------------------------------------------------------------- geometry\n    @property\n    def L1(self) -> int:\n        return int(self.box[0])\n\n    @property\n    def L2(self) -> int:\n        return int(self.box[1])\n\n    @property\n    def L3(self) -> int:\n        return int(self.box[2]) if self.crystal.D == 3 else 1\n\n    @property\n    def b(self) -> int:\n        return self.crystal.b\n\n    @property\n    def shape(self) -> tuple[int, int, int, int]:\n        return (self.b, self.L3, self.L2, self.L1)\n\n    def site_mask(self) -> np.ndarray:\n        return np.ones(self.shape, bool) if self.mask is None else self.mask.astype(bool)\n\n    @property\n    def n_active(self) -> int:\n        return int(self.site_mask().sum())\n\n    def bond_matrix(self, i: int) -> np.ndarray:\n        bd = self.crystal.bonds[i]\n        M = self.J.get(bd.shell, 0.0) * np.eye(self.n)\n        if i in self.extra:\n            M = M + self.extra[i]\n        return M\n\n    def colour_map(self) -> np.ndarray:\n        """Colour of every site, shape (b, L3, L2, L1)."""\n        out = np.zeros(self.shape, np.int64)\n        for a in range(self.b):\n            for z in range(self.L3):\n                for y in range(self.L2):\n                    for x in range(self.L1):\n                        c = (x, y, z)[: self.crystal.D]\n                        out[a, z, y, x] = self.colouring.colour(c, a)\n        return out\n\n\n# -------------------------------------------------------------------- matrix\ndef ising(crystal: str, L: int, J: float = 1.0, shells=(1,), T_ref=None) -> System:\n    c = cr.make(crystal, tuple(shells))\n    return System(f"Ising {crystal}", 1, c, (L,) * c.D, (True,) * c.D, {s: J for s in shells}, T_ref=T_ref)\n\n\ndef xy(crystal: str, L: int, J: float = 1.0, T_ref=None) -> System:\n    c = cr.make(crystal)\n    return System(f"XY {crystal}", 2, c, (L,) * c.D, (True,) * c.D, {1: J}, T_ref=T_ref)\n\n\ndef heisenberg(crystal: str, L: int, J: float = 1.0, T_ref=None, label=None) -> System:\n    c = cr.make(crystal)\n    return System(label or f"Heisenberg {crystal}", 3, c, (L,) * c.D, (True,) * c.D, {1: J}, T_ref=T_ref)\n\n\ndef nanodot(Rd: float = 18.30, L: int = 5, Jex=1.0, Kan1=0.177, Kan2=0.034, KanS=0.139, Hex=0.483,\n            KDM=0.88, gamma_deg=90.0, shells=(1,), J_shells=None) -> System:\n    """The nanodot system: sc Heisenberg disc inside an open box.\n\n    `shells` adds sc exchange shells beyond the first, with couplings `J_shells` {shell: J} relative\n    to Jex; the colouring search then finds a colouring admissible for all of them. DMI stays on the\n    first-shell in-plane bonds and the surface is still defined by first neighbours.\n\n    The disc never reaches the box edge, so every seam bond would touch a masked (zero) site; an\n    open box is the same Hamiltonian as a periodic one and meets condition (8) without a constraint on N.\n\n    Interfacial DMI on in-plane bonds (the local field term\n    KDM * (-dyz, dxz, dyx - dxy)), cubic anisotropy with surface constant KanS where a site has\n    fewer than six in-disc neighbours, and a Zeeman field Hex (cos g, 0, sin g).\n    """\n    shells = tuple(sorted(set(shells) | {1}))\n    c = cr.make("sc", shells)\n    J = {1: Jex, **{int(k): float(v) for k, v in (J_shells or {}).items() if int(k) in shells and int(k) != 1}}\n    L2 = int(Rd) + 1\n    N = 2 * L2 + 1\n    xs = np.arange(-L2, L2 + 1)\n    xg, yg = np.meshgrid(xs, xs, indexing="xy")\n    disk = (xg ** 2 + yg ** 2) <= Rd ** 2               # [y, x]\n    mask = np.broadcast_to(disk, (L, N, N)).copy()[None]  # (1, L, N, N)\n    surface = np.zeros_like(mask)\n    for z in range(L):\n        for y in range(N):\n            for x in range(N):\n                if not mask[0, z, y, x]:\n                    continue\n                k = 0\n                for dz, dy, dx in ((0, 0, 1), (0, 0, -1), (0, 1, 0), (0, -1, 0), (1, 0, 0), (-1, 0, 0)):\n                    nz, ny, nx = z + dz, (y + dy) % N, (x + dx) % N\n                    if 0 <= nz < L and mask[0, nz, ny, nx]:\n                        k += 1\n                surface[0, z, y, x] = k < 6\n    k1 = np.where(surface, KanS, Kan1).astype(np.float32) * mask\n    extra = {}\n    for i, bd in enumerate(c.bonds):\n        dx, dy, dz = bd.delta\n        E = np.zeros((3, 3))\n        if (dx, dy, dz) == (1, 0, 0):\n            E[1, 2], E[2, 1] = KDM, -KDM\n        elif (dx, dy, dz) == (-1, 0, 0):\n            E[1, 2], E[2, 1] = -KDM, KDM\n        elif (dx, dy, dz) == (0, 1, 0):\n            E[0, 2], E[2, 0] = -KDM, KDM\n        elif (dx, dy, dz) == (0, -1, 0):\n            E[0, 2], E[2, 0] = KDM, -KDM\n        if E.any():\n            extra[i] = E\n    g = np.deg2rad(gamma_deg)\n    return System("Heisenberg nanodot", 3, c, (N, N, L), (False, False, False), J, extra,\n                  h=(Hex * float(np.cos(g)), 0.0, Hex * float(np.sin(g))), cubic_k1=k1, cubic_k2=Kan2,\n                  mask=mask)\n\n\n# Reference critical temperatures (J = kB = 1), from experiments v03 section E4.\nTC = {\n    "Ising square": 2.0 / np.log(1 + np.sqrt(2)),\n    "Ising triangular": 4.0 / np.log(3.0),\n    "Ising sc": 4.511523,      # 1/0.22165463, Ferrenberg, Xu and Landau 2018\n    "Ising bcc": 6.355,        # 1/0.157373, Lundow and Campbell 2018\n    "Ising fcc": 9.797132,     # 1/0.1020707(2), Yu 2015 (Physica A 419, 75)\n    "XY sc": 2.20184,          # 1/0.4541652, Xu et al. 2019\n    "Heisenberg sc": 1.4430,   # Chen, Ferrenberg and Landau 1993\n    "Heisenberg bcc": 2.0542,  # Chen, Ferrenberg and Landau 1993\n}\n\n\ndef matrix(small: bool = True) -> list[System]:\n    """The 13 systems of experiment E1 at a small size per crystal (seam-compatible boxes)."""\n    s = [\n        ising("square", 8, T_ref=TC["Ising square"]),\n        ising("triangular", 6, T_ref=TC["Ising triangular"]),\n        ising("sc", 4, T_ref=TC["Ising sc"]),\n        ising("bcc", 4, T_ref=TC["Ising bcc"]),\n        ising("fcc", 4, T_ref=TC["Ising fcc"]),\n        xy("square", 8, T_ref=0.893),\n        xy("sc", 4, T_ref=TC["XY sc"]),\n        heisenberg("sc", 4, T_ref=TC["Heisenberg sc"]),\n        heisenberg("bcc", 4, T_ref=TC["Heisenberg bcc"]),\n        heisenberg("fcc", 4, T_ref=None),\n        heisenberg("triangular", 6, J=-1.0, T_ref=0.285, label="Heisenberg triangular AFM"),\n        heisenberg("kagome", 6, J=-1.0, T_ref=None, label="Heisenberg kagome AFM"),\n        nanodot(Rd=3.0, L=3),\n    ]\n    return s\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "f3755bd076a9c0cd28a3da7dc7c774d2db5188f5e84085dce981f276c33fde03", "system.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/system.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/system.py  sha256 f3755bd076a9c0cd…")

In [ ]:
# Spin_Colour_Kernel/generator.py — byte-identical to the repository. sha256 5e9058a6ce269efcd6b61030bc90d836d2853b18bf683f7ae06e0da598241694
import hashlib, os
_src = '"""The kernel generator (proposal r07, section 5): one System -> one specialized Triton kernel.\n\nSpins are read in the storage type of `src_ptr` (int8 for Ising, float32 for vector models) and\nconverted to float32 for the arithmetic; results are stored back in the type of `dst_ptr`.\n\nThe model (n, proposal, anisotropy), the bond table, the colouring and the box are emitted as\nstraight-line Triton source; only the replica index, the inverse temperature and the counter state\nare runtime values. One launch updates one (colour, sublattice) class; a sweep is the composition\nof the n_c * b class launches, in a fixed order.\n\nColour compaction. Along x a linear colouring repeats with period P = lcm_r m_r / gcd(K_r1, m_r)\n(STRIDE = P), and along a row each colour of the row\'s coset appears at most once per period. A class\nlaunch therefore gives each program a tile of R rows x BX = ceil(L1 / P) sites: each lane solves its\nrow\'s residue x0 from the colouring and takes x0 + P * j. A row without the colour does nothing, and\nwhen P is close to L1 one program still covers R rows instead of a single site. Every load of a class launch is masked by the update predicate, so a sweep reads the\nlattice about once whatever n_c is, instead of n_c times. The sequential reference and the synchronous control keep STRIDE = 1, R = 1.\n\nRandomness is Philox (`tl.rand4x`) keyed by a per-sweep seed and countered by (replica, site), so the\nstream is a pure function of the site and the sweep, never of the launch grid. The same compiled\nkernel serves the parallel chain and the sequential reference: the reference launches it once per\nsite (`x_only`, `row0`), the parallel chain once per class.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.util\nimport math\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .system import System\n\nCACHE = Path(os.environ.get("SCK_CACHE", Path.home() / ".cache" / "spin_colour_kernel"))\n\n\ndef _f(x: float) -> str:\n    return repr(float(np.float32(x)))\n\n\ndef _coord_expr(var: str, d: int, L: int, periodic: bool) -> tuple[str, str | None]:\n    """Neighbour coordinate and its validity predicate (None when always valid)."""\n    if d == 0:\n        return var, None\n    if periodic:\n        return f"(({var} + {d % L}) % {L})", None\n    e = f"({var} + {d})"\n    return e, f"(({e} >= 0) & ({e} < {L}))"\n\n\ndef x_period(col) -> int:\n    """Period along x of a linear colouring: lcm over its factors of m_r / gcd(K_r1, m_r)."""\n    P = 1\n    for r, mr in enumerate(col.m):\n        P = math.lcm(P, mr // math.gcd(int(col.K[r][0]) % mr or mr, mr))\n    return P\n\n\ndef _colour_expr(sys_: System, xv: str, a: int) -> str:\n    """Colour of site (xv, y, z) of sublattice a, as a Triton expression (mixed-radix, eq. 6)."""\n    col, D = sys_.colouring, sys_.crystal.D\n    if not col.m:\n        return f"{xv} * 0"\n    terms = []\n    for r, mr in enumerate(col.m):\n        coeffs = list(col.K[r]) + [0] * (3 - D)\n        terms.append((f"({coeffs[0]} * {xv} + {coeffs[1]} * y + {coeffs[2]} * z + {col.o[a][r]}) % {mr}", mr))\n    expr = terms[0][0]\n    for e, mr in terms[1:]:\n        expr = f"({expr}) * {mr} + {e}"\n    return expr\n\n\ndef _emit_tile(w, sys_: System, chunked: bool) -> None:\n    """Lanes, rows and the colour-compacted x coordinate of one class launch; defines live, xc, yv, zv.\n\n    chunked: one-row programs take their x chunk from grid axis 2 (rows longer than one program\'s lanes)."""\n    b, L1, L2, L3 = sys_.b, sys_.L1, sys_.L2, sys_.L3\n    # a program covers a tile of R rows x BX sites, flattened onto BLOCK lanes: lane l is row l // BX\n    # of the tile and the (l % BX)-th site of that row\'s colour class (x0 + STRIDE * j)\n    w("    lane = tl.arange(0, BLOCK)")\n    w("    if R == 1:")                       # one row per program: row coordinates stay scalars\n    w("        j = lane + tl.program_id(2) * BLOCK" if chunked else "        j = lane")\n    w("        row = tl.program_id(1) + row0")\n    w(f"        z = row // {L2}")\n    w(f"        y = row % {L2}")\n    w("    else:")\n    w("        j = lane % BX")\n    w("        rowv = tl.program_id(1) * R + lane // BX + row0")\n    w(f"        rok = (lane < R * BX) & (rowv < {L3 * L2})")\n    w("        rowc = tl.where(rok, rowv, 0)")\n    w(f"        z = rowc // {L2}")\n    w(f"        y = rowc % {L2}")\n\n    # compaction: along a row the colour has period STRIDE and each colour of the row\'s coset appears\n    # at most once per period, so each lane finds its row\'s residue x0 (L1 when the row lacks it)\n    w("    if STRIDE > 1:")\n    w(f"        x0 = tl.zeros([BLOCK], dtype=tl.int32) + {L1}")\n    w("        for q in tl.static_range(STRIDE):")\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"            {kw} ALPHA == {a}:")\n        w(f"                cq = {_colour_expr(sys_, \'q\', a)}")\n        w("                x0 = tl.where(cq == COLOUR, q, x0)")\n    w("    else:")\n    w("        x0 = 0")\n    w("    x = x0 + STRIDE * j")\n    w("    xo = tl.zeros([BLOCK], dtype=tl.int32) + x_only")\n    w("    if R == 1:")\n    w(f"        live = (x < {L1}) & ((x == xo) | (xo < 0))")\n    w("    else:")\n    w(f"        live = rok & (x < {L1}) & ((x == xo) | (xo < 0))")\n    w("    xc = tl.where(live, x, 0)")\n    w("    yv = tl.zeros([BLOCK], dtype=tl.int32) + y")\n    w("    zv = tl.zeros([BLOCK], dtype=tl.int32) + z")\n\n\ndef _emit_class(w, sys_: System) -> None:\n    """Update predicate of one (colour, sublattice) class: defines colour, upd, own, m_own (masked systems), ld."""\n    b, L1, L2 = sys_.b, sys_.L1, sys_.L2\n    pl = sys_.L3 * L2 * L1\n    has_mask = sys_.mask is not None\n    # colour mask, from (6), per alpha\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        w(f"        colour = {_colour_expr(sys_, \'xc\', a)}")\n    w("    if COLOUR < 0:")\n    w("        upd = live")\n    w("    else:")\n    w("        upd = live & (colour == COLOUR)")\n    w(f"    own = ALPHA * {pl} + z * {L2 * L1} + y * {L1} + xc")\n    if has_mask:\n        w("    m_own = tl.load(mask_ptr + own, mask=live, other=0)")\n        w("    upd = upd & (m_own != 0)")\n    # a class launch reads only the sites it updates; the synchronous control reads every live site\n    w("    if SYNC:")\n    w("        ld = live")\n    w("    else:")\n    w("        ld = upd")\n\n\ndef source(sys_: System, packed: bool = False, bitwise: bool = False, independent: bool = False) -> str:\n    """Triton source text for one system.\n\n    bitwise (Ising only): true multi-spin coding, see `bitwise.source`; same packed storage, but the\n    update stays in bit operations with one uniform per (word, site, sweep) and a host acceptance table.\n    independent (bitwise only): one 32-bit uniform per replica instead, compared bit-sliced.\n\n    packed (Ising only): multi-spin coding over replicas. Word w of a site holds that site\'s spin in\n    replicas 32 w .. 32 w + 31, one bit each (1 = +1). A program loads one word per neighbour, unpacks it\n    into a [BLOCK, 32] tile, runs the same Metropolis step with the same Philox counter per (replica,\n    site) as the unpacked kernel, and packs the result back with an OR. Rows longer than BLOCK are split\n    over grid axis 2."""\n    if independent and not bitwise:\n        raise ValueError("independent per-replica draws need the bitwise kernel")\n    if bitwise:\n        from .bitwise import source as bitwise_source\n        return bitwise_source(sys_, independent)\n    n, b = sys_.n, sys_.b\n    if packed and n != 1:\n        raise ValueError("bit packing is for Ising (n = 1) only")\n    P = packed\n    RB = "BLOCK, 32" if P else "BLOCK"\n    L1, L2, L3 = sys_.L1, sys_.L2, sys_.L3\n    D = sys_.crystal.D\n    per = list(sys_.periodic) + [False] * (3 - D)\n    pl = L3 * L2 * L1                      # one (component, alpha) plane\n    nsite = b * pl\n    has_mask = sys_.mask is not None\n    has_cubic = sys_.cubic_k1 is not None\n    lines: list[str] = []\n    w = lines.append\n    w("import triton")\n    w("import triton.language as tl")\n    w("")\n    w("")\n    if P:\n        w("@triton.jit")\n        w("def _or(a, b):")\n        w("    return a | b")\n        w("")\n        w("")\n    w(\'@triton.jit(do_not_specialize=["x_only", "row0"])\')\n    w("def kernel(src_ptr, dst_ptr, mask_ptr, k1_ptr, beta_ptr, state_ptr, x_only, row0,")\n    w("           ALPHA: tl.constexpr, COLOUR: tl.constexpr, BLOCK: tl.constexpr, SYNC: tl.constexpr,")\n    w("           STRIDE: tl.constexpr, BX: tl.constexpr, R: tl.constexpr):")\n    if P:\n        w("    wrd = tl.program_id(0)")\n        w("    rr = tl.arange(0, 32)")\n        w("    rep = wrd * 32 + rr")\n    else:\n        w("    rep = tl.program_id(0)")\n    _emit_tile(w, sys_, True)     # one-row programs may be chunked over grid axis 2 (packed: PTILE; float: MAX_BLOCK)\n    w("    beta = tl.load(beta_ptr + rep)")\n    w("    base_seed = tl.load(state_ptr + 0)")\n    w("    sweep = tl.load(state_ptr + 1)")\n    w("    seed = ((base_seed * 1000003 + sweep * 7919 + 12345) % 2147483629).to(tl.int32)")\n    w(f"    rbase = wrd.to(tl.int64) * {nsite}" if P else f"    rbase = rep.to(tl.int64) * {n * nsite}")\n    _emit_class(w, sys_)\n    # own spin\n    if P:\n        w("    w0 = tl.load(src_ptr + rbase + own, mask=ld, other=0)")\n        w("    s0 = (((w0[:, None] >> rr[None, :]) & 1) * 2 - 1).to(tl.float32)")\n    else:\n        for k in range(n):\n            w(f"    s{k} = tl.load(src_ptr + rbase + {k * nsite} + own, mask=ld, other=0).to(tl.float32)")\n    # local field, per alpha\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        groups: dict[float, list[int]] = {}\n        extras: list[int] = []\n        for i, bd in enumerate(sys_.crystal.bonds):\n            if bd.alpha != a:\n                continue\n            groups.setdefault(float(sys_.J.get(bd.shell, 0.0)), []).append(i)\n            if i in sys_.extra:\n                extras.append(i)\n        loaded = {}\n        for k in range(n):\n            w(f"        v{k} = tl.zeros([{RB}], dtype=tl.float32)")\n        idx = 0\n        for Jv, members in groups.items():\n            if Jv == 0.0 and not any(i in sys_.extra for i in members):\n                continue\n            for k in range(n):\n                w(f"        g{k} = tl.zeros([{RB}], dtype=tl.float32)")\n            for i in members:\n                bd = sys_.crystal.bonds[i]\n                d = list(bd.delta) + [0] * (3 - D)\n                ex, vx = _coord_expr("xc", d[0], L1, per[0])\n                ey, vy = _coord_expr("yv", d[1], L2, per[1])\n                ez, vz = _coord_expr("zv", d[2], L3, per[2])\n                valid = " & ".join(["ld"] + [p for p in (vx, vy, vz) if p])\n                w(f"        q = {bd.alpha2 * pl} + {ez} * {L2 * L1} + {ey} * {L1} + {ex}")\n                w(f"        ok = {valid}")\n                w("        q = tl.where(ok, q, 0)")\n                if P and has_mask:      # a masked neighbour holds no spin: it contributes 0, as in the unpacked store\n                    w("        ok = ok & (tl.load(mask_ptr + q, mask=ok, other=0) != 0)")\n                for k in range(n):\n                    if P:\n                        w("        tw = tl.load(src_ptr + rbase + q, mask=ok, other=0)")\n                        w("        t = tl.where(ok[:, None], (((tw[:, None] >> rr[None, :]) & 1) * 2 - 1).to(tl.float32), 0.0)")\n                    else:\n                        w(f"        t = tl.load(src_ptr + rbase + {k * nsite} + q, mask=ok, other=0).to(tl.float32)")\n                    w(f"        g{k} = g{k} + t")\n                    if i in sys_.extra:\n                        w(f"        n{idx}_{k} = t")\n                if i in sys_.extra:\n                    loaded[i] = idx\n                    idx += 1\n            if Jv != 0.0:\n                for k in range(n):\n                    w(f"        v{k} = v{k} + {_f(Jv)} * g{k}")\n        for i in extras:\n            E = sys_.extra[i]\n            j = loaded[i]\n            for k in range(n):\n                terms = [f"{_f(E[k, l])} * n{j}_{l}" for l in range(n) if E[k, l] != 0.0]\n                if terms:\n                    w(f"        v{k} = v{k} + ({\' + \'.join(terms)})")\n    if sys_.h is not None:\n        for k in range(n):\n            if sys_.h[k] != 0.0:\n                w(f"    v{k} = v{k} + {_f(sys_.h[k])}")\n    # proposal (9)\n    w(f"    off = (rep[None, :] * {nsite} + own[:, None]).to(tl.int32)" if P else f"    off = (rep * {nsite} + own).to(tl.int32)")\n    w("    u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    if n == 1:\n        w("    t0 = -s0")\n    elif n == 2:\n        w("    phi = 6.283185307179586 * u1")\n        w("    t0 = tl.cos(phi)")\n        w("    t1 = tl.sin(phi)")\n    else:\n        w("    cz = 2.0 * u1 - 1.0")\n        w("    phi = 6.283185307179586 * u2")\n        w("    rr = tl.sqrt(tl.maximum(1.0 - cz * cz, 0.0))")\n        w("    t0 = rr * tl.cos(phi)")\n        w("    t1 = rr * tl.sin(phi)")\n        w("    t2 = cz")\n    # energy change (3)\n    dE = " + ".join(f"(t{k} - s{k}) * v{k}" for k in range(n))\n    w(f"    dE = -({dE})")\n    if has_cubic:\n        w("    k1 = tl.load(k1_ptr + own, mask=ld, other=0.0)")\n        k2 = _f(sys_.cubic_k2)\n        for p in ("t", "s"):\n            w(f"    a_{p} = k1 * (({p}0 * {p}1) * ({p}0 * {p}1) + ({p}0 * {p}2) * ({p}0 * {p}2) "\n              f"+ ({p}1 * {p}2) * ({p}1 * {p}2)) + {k2} * ({p}0 * {p}1 * {p}2) * ({p}0 * {p}1 * {p}2)")\n        w("    dE = dE + (a_t - a_s)")\n    bt, ud = ("beta[None, :]", "upd[:, None]") if P else ("beta", "upd")\n    if sys_.acceptance == "barker":\n        # heat-bath (Barker) rule: accept with 1 / (1 + exp(beta dE)); dE = 0 is accepted with probability 1/2\n        w(f"    acc = (u0 * (1.0 + tl.exp({bt} * dE)) < 1.0) & {ud}")\n    else:\n        w(f"    acc = (u0 < tl.exp(-{bt} * dE)) & {ud}")\n    if P:\n        w("    up = (tl.where(acc, t0, s0) > 0).to(tl.int32) << rr[None, :]")\n        w("    r0 = tl.reduce(up, 1, _or)")\n    else:\n        for k in range(n):\n            w(f"    r{k} = tl.where(acc, t{k}, s{k}).to(dst_ptr.dtype.element_ty)")\n    w("    if SYNC:")\n    wm = "live & (m_own != 0)" if has_mask else "live"\n    for k in range(n):\n        w(f"        tl.store(dst_ptr + rbase + {k * nsite} + own, r{k}, mask={wm})")\n    w("    else:")\n    for k in range(n):\n        w(f"        tl.store(dst_ptr + rbase + {k * nsite} + own, r{k}, mask=upd)")\n    return "\\n".join(lines) + "\\n"\n\n\ndef compile_system(sys_: System, packed: bool = False, bitwise: bool = False, independent: bool = False):\n    """Write the generated source to the cache, import it, and return the jitted kernel."""\n    src = source(sys_, packed, bitwise, independent)\n    key = hashlib.sha256(src.encode()).hexdigest()[:16]\n    CACHE.mkdir(parents=True, exist_ok=True)\n    path = CACHE / f"sck_{key}.py"\n    if not path.exists():\n        path.write_text(src)\n    name = f"sck_{key}"\n    if name in sys.modules:\n        return sys.modules[name].kernel, src\n    spec = importlib.util.spec_from_file_location(name, path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[name] = mod\n    spec.loader.exec_module(mod)\n    return mod.kernel, src\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "5e9058a6ce269efcd6b61030bc90d836d2853b18bf683f7ae06e0da598241694", "generator.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/generator.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/generator.py  sha256 5e9058a6ce269efc…")

In [ ]:
# Spin_Colour_Kernel/bitwise.py — byte-identical to the repository. sha256 a4488d0ada766a05b1381a32f9744a867849e8547d40e8d9c6c03d36822f225d
import hashlib, os
_src = '"""Bitwise multi-spin-coded Ising: asynchronous multi-spin coding over replicas (routes B and C).\n\nStorage is the packed word layout of `generator.source(packed=True)`: word w of a site holds that site\'s\nspin in replicas 32 w .. 32 w + 31, one bit each (1 = +1). Unlike the packed path, which unpacks every word\nto a [BLOCK, 32] float tile, this kernel never leaves the bits:\n\n* per neighbour j present in the box and the mask, x = s_i XOR s_j marks the anti-aligned replicas, and a\n  ripple adder accumulates x into NP bit planes, so plane p holds bit p of the anti-aligned count k of every\n  replica (NP = bit length of the largest coordination, 4 for fcc);\n* the energy change of flipping s_i is dE = 2 J (z_eff - 2 k) + 2 h s_i, with z_eff the number of present\n  neighbours of the site (the same for all 32 replicas), so the acceptance probability is a host table\n  p[word, s, z_eff, k] built per temperature for the system\'s rule (Metropolis min(1, e^{-beta dE}) or\n  Barker 1 / (1 + e^{beta dE}));\n* one uniform u per (word, site, sweep) is shared by the 32 replicas of the word, and the flip mask is\n  OR_k [(u < p(z_eff, k[, s])) ? eq_k : 0], with eq_k the replicas whose count equals k.\n\nEach replica is still an exact Metropolis or Barker chain; replicas of one word are correlated through\nthe shared uniform (and may coalesce), so independent samples come from different words.\n\nRandomness: Philox keyed by the per-sweep seed of the other kernels. Uniform (word w, site i) is output\nw % 4 of the call with counter (w // 4) * nsite + i, so one call serves four words and a program that\ncovers an aligned group of four words wastes no output. The stream is a function of (word, site, sweep)\nonly, never of the launch grid; `uniforms` reproduces it in NumPy for the sequential reference.\n\nIndependent mode (`source(sys_, independent=True)`, `Sampler(..., bitwise=True, independent=True)`): the\nshared uniform makes the replicas of a word coalesce (at Tc on the square lattice 32 replicas collapse to two\nconfigurations within about 50 sweeps), so this mode gives every replica (bit lane) its own draw and keeps the\nupdate exact and bitwise:\n\n* the table holds P = floor(p 2^32) as an integer in [0, 2^32] (`p_fixed`); a replica flips iff its 32-bit\n  uniform integer U satisfies U < P, an exact Bernoulli(P / 2^32) draw. p is therefore quantized down to a\n  multiple of 2^-32; p = 1 gives P = 2^32 (always, no random bits) and P = 0 never flips;\n* U is compared bit-sliced over the 32 lanes, most significant bit first: at level l a random word R_l holds\n  bit 31 - l of U of every lane and Pb_l the same bit of every lane\'s P (selected by the count planes, and by\n  s when h != 0); accept |= und & ~R_l & Pb_l, und &= ~(R_l ^ Pb_l). Lanes start undecided iff 0 < P < 2^32;\n  each level halves them, so the program exits as soon as its whole tile is decided, and after 32 levels the\n  rest (U == P) is rejected, exactly;\n* R_{4g} .. R_{4g+3} of (word w, site i) are the four raw outputs of `tl.randint4x(key_g, w nsite + i)` with\n  the int32 counter of the shared mode and the per-group key key_g = (seed + (g + 1) GROUP_KEY_STEP) mod\n  2147483629, a function of (word, site, level group, sweep) only; `lane_uniforms` rebuilds every replica\'s U\n  in NumPy. (A 64-bit offset would switch Triton\'s GPU randint4x to 64-bit Philox, which the interpreter does\n  not do, so the level group goes into the key, not into the counter.)\n\nOnly systems with one Ising coupling J on every bond that carries one (any crystal, any shells of equal J),\nan optional uniform field and an optional site mask qualify; `check` refuses the others.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .system import System\n\n#: replicas per word\nWORD = 32\n\n\n# ------------------------------------------------------------------ system\ndef check(sys_: System) -> tuple[float, float, int]:\n    """(J, h, Z) of a system the bitwise kernel can run, Z the largest coordination; ValueError otherwise."""\n    if sys_.n != 1:\n        raise ValueError("bitwise multi-spin coding is for Ising (n = 1) only")\n    if sys_.extra:\n        raise ValueError("bitwise multi-spin coding needs isotropic exchange only (no extra bond matrices)")\n    if sys_.cubic_k1 is not None:\n        raise ValueError("bitwise multi-spin coding does not support anisotropy")\n    Js = {float(sys_.J.get(bd.shell, 0.0)) for bd in sys_.crystal.bonds} - {0.0}\n    if len(Js) != 1:\n        raise ValueError(f"bitwise multi-spin coding needs one uniform coupling J on every bond, got {sorted(Js)}")\n    h = float(sys_.h[0]) if sys_.h is not None else 0.0\n    return Js.pop(), h, max(len(bonds(sys_, a)) for a in range(sys_.b))\n\n\ndef bonds(sys_: System, a: int) -> list:\n    """Bonds leaving sublattice a that carry a coupling, in bond-table order."""\n    return [bd for bd in sys_.crystal.bonds if bd.alpha == a and sys_.J.get(bd.shell, 0.0) != 0.0]\n\n\ndef p_table(sys_: System, beta: np.ndarray) -> np.ndarray:\n    """Acceptance probabilities p[word, s, z_eff, k], float32, for one inverse temperature per word.\n\n    s = 0 for spin -1 and 1 for +1, z_eff the number of present neighbours, k <= z_eff the anti-aligned\n    ones; entries with k > z_eff are never read and hold 0."""\n    return _p64(sys_, beta).astype(np.float32)\n\n\ndef p_fixed(sys_: System, beta: np.ndarray) -> np.ndarray:\n    """Independent mode: P = floor(p 2^32) in [0, 2^32], int64, same layout as `p_table` (p in float64).\n\n    A replica with 32-bit uniform integer U flips iff U < P: probability P / 2^32, i.e. p rounded down to a\n    multiple of 2^-32 (P = 2^32 when p = 1: always; P = 0: never)."""\n    P = np.floor(_p64(sys_, beta) * 2.0 ** 32).astype(np.int64)\n    return np.clip(P, 0, 2 ** 32)\n\n\ndef _p64(sys_: System, beta: np.ndarray) -> np.ndarray:\n    J, h, Z = check(sys_)\n    beta = np.asarray(beta, np.float64).reshape(-1)\n    p = np.zeros((beta.size, 2, Z + 1, Z + 1), np.float64)\n    for s in (0, 1):\n        sigma = 2 * s - 1\n        for ze in range(Z + 1):\n            k = np.arange(ze + 1)\n            dE = 2.0 * J * (ze - 2 * k) + 2.0 * h * sigma\n            x = beta[:, None] * dE[None, :]\n            if sys_.acceptance == "barker":\n                p[:, s, ze, : ze + 1] = 0.5 * (1.0 - np.tanh(0.5 * x))\n            else:\n                p[:, s, ze, : ze + 1] = np.exp(-np.maximum(x, 0.0))\n    return p\n\n\n# ------------------------------------------------------------------ randomness\n_M = 0xFFFFFFFF\n\n\ndef philox_uniform4(seed: int, counters) -> np.ndarray:\n    """NumPy twin of Triton\'s `tl.rand4x(seed, counters)` for an int32 seed and int32 counters: (4, N) float32."""\n    out = philox_raw4(seed, counters).view(np.int32)\n    out = np.where(out < 0, ~out, out)\n    return out.astype(np.float32) * np.float32(4.6566127342e-10)\n\n\ndef philox_raw4(seed: int, c0, c1=0, rounds: int = 10) -> np.ndarray:\n    """NumPy twin of Triton\'s `tl.randint4x(seed, off)` with off = c1 2^32 + c0 (c0 < 2^32): (4, N) uint32.\n\n    Philox-4x32-10 with key (seed low, seed high) and counter (c0, c1, 0, 0)."""\n    c0 = np.asarray(c0, np.int64).reshape(-1)\n    c1 = np.broadcast_to(np.asarray(c1, np.int64).reshape(-1), c0.shape)\n    c0, c1 = (c0 & _M).astype(np.uint64), (c1 & _M).astype(np.uint64)\n    c2 = np.zeros_like(c0)\n    c3 = np.zeros_like(c0)\n    k0 = np.uint64(int(seed) & _M)\n    k1 = np.uint64((int(seed) >> 32) & _M)\n    A, B = np.uint64(0xD2511F53), np.uint64(0xCD9E8D57)\n    m, s32 = np.uint64(_M), np.uint64(32)\n    for _ in range(rounds):\n        p0, p2 = B * c2, A * c0           # < 2^64: exact in uint64\n        c0, c1, c2, c3 = ((p0 >> s32) ^ c1 ^ k0) & m, p0 & m, ((p2 >> s32) ^ c3 ^ k1) & m, p2 & m\n        k0 = (k0 + np.uint64(0x9E3779B9)) & m\n        k1 = (k1 + np.uint64(0xBB67AE85)) & m\n    return np.stack([c0, c1, c2, c3]).astype(np.uint32)\n\n\ndef sweep_seed(base_seed: int, sweep: int) -> int:\n    """The per-sweep Philox key of every generated kernel."""\n    return ((int(base_seed) * 1000003 + int(sweep) * 7919 + 12345) % 2147483629)\n\n\ndef uniforms(base_seed: int, sweep: int, n_words: int, nsite: int) -> np.ndarray:\n    """The kernel\'s uniform of every (word, site) in one sweep: shape (n_words, nsite), float32."""\n    groups = -(-n_words // 4)\n    ctr = np.arange(groups, dtype=np.int64)[:, None] * nsite + np.arange(nsite, dtype=np.int64)[None, :]\n    u = philox_uniform4(sweep_seed(base_seed, sweep), ctr.reshape(-1)).reshape(4, groups, nsite)\n    return u.transpose(1, 0, 2).reshape(groups * 4, nsite)[:n_words]\n\n\n#: independent mode: random words per (word, site, sweep), one per bit of U, four per Philox call\nLEVELS = 32\n#: independent mode: the Philox key of level group g is (seed + (g + 1) GROUP_KEY_STEP) mod 2147483629\nGROUP_KEY_STEP = 104729\n#: independent mode: Philox-4x32 rounds per call, Triton\'s default 10 (the kernel and the NumPy twin agree); 7 rounds\n#: is about 15% faster and is kept as an ablation (experiments v11/v12), not as the default\nINDEPENDENT_ROUNDS = 10\n\n\ndef group_key(seed: int, g: int) -> int:\n    return (int(seed) + (int(g) + 1) * GROUP_KEY_STEP) % 2147483629\n\n\ndef lane_uniforms(base_seed: int, sweep: int, n_words: int, nsite: int) -> np.ndarray:\n    """Independent mode: the 32-bit uniform integer U of every (replica, site) in one sweep, (32 n_words, nsite)\n    int64. Bit 31 - l of U of replica 32 w + j is bit j of the random word R_l(w, site), R_{4g + q} being\n    output q of the Philox call with key `group_key(seed, g)` and counter w nsite + site."""\n    ctr = (np.arange(n_words, dtype=np.int64)[:, None] * nsite + np.arange(nsite, dtype=np.int64)[None, :]).reshape(-1)\n    seed = sweep_seed(base_seed, sweep)\n    lanes = np.arange(WORD, dtype=np.uint64)\n    U = np.zeros((n_words, WORD, nsite), np.int64)\n    for g in range(LEVELS // 4):\n        R = philox_raw4(group_key(seed, g), ctr, rounds=INDEPENDENT_ROUNDS).astype(np.uint64).reshape(4, n_words, 1, nsite)\n        for q in range(4):\n            bit = ((R[q] >> lanes[None, :, None]) & np.uint64(1)).astype(np.int64)     # (n_words, 32, nsite)\n            U |= bit << (LEVELS - 1 - (4 * g + q))\n    return U.reshape(n_words * WORD, nsite)\n\n\n# ------------------------------------------------------------------ sequential reference\ndef classes(sys_: System) -> list[tuple[int, int]]:\n    """(colour, sublattice) classes in sweep order: colour-major, then sublattice; empty classes skipped."""\n    cm, m = sys_.colour_map(), sys_.site_mask()\n    return [(c, a) for c in range(sys_.colouring.n_colours) for a in range(sys_.b) if np.any((cm[a] == c) & m[a])]\n\n\ndef reference_sweep(sys_: System, spins: np.ndarray, ptab: np.ndarray, base_seed: int, sweep: int,\n                    independent: bool = False) -> np.ndarray:\n    """One sequential sweep of every replica on unpacked spins, no bit operations.\n\n    spins: (B, b, L3, L2, L1) of +-1 (0 on masked sites). For each class in sweep order and each of its sites\n    in raster order, every replica r computes its local sum sigma_i sum_j s_j over the present neighbours,\n    k = (z_eff - that sum) / 2, draws the uniform of (word r // 32, site, sweep) and flips when\n    u < ptab[r // 32, s, z_eff, k]. independent: ptab is the integer table of `p_fixed` and replica r flips iff\n    its own integer U (`lane_uniforms`) satisfies U < ptab[r // 32, s, z_eff, k]. Returns the updated copy."""\n    _, h, _ = check(sys_)\n    S = np.array(spins, dtype=np.int64, copy=True)\n    B = S.shape[0]\n    b, L3, L2, L1 = sys_.shape\n    D = sys_.crystal.D\n    box = (L1, L2, L3)\n    per = list(sys_.periodic) + [False] * (3 - D)\n    m = sys_.site_mask()\n    cm = sys_.colour_map()\n    nsite = b * L3 * L2 * L1\n    W = -(-B // WORD)\n    if independent:\n        UL = lane_uniforms(base_seed, sweep, W, nsite)\n    else:\n        U = uniforms(base_seed, sweep, W, nsite)\n    word = np.arange(B) // WORD\n    plane = 1 if h != 0.0 else None\n    for c, a in classes(sys_):\n        for z, y, x in np.argwhere((cm[a] == c) & m[a]):\n            nb = []\n            for bd in bonds(sys_, a):\n                d = list(bd.delta) + [0] * (3 - D)\n                q = [x + d[0], y + d[1], z + d[2]]\n                ok = True\n                for k in range(3):\n                    if per[k]:\n                        q[k] %= box[k]\n                    elif not 0 <= q[k] < box[k]:\n                        ok = False\n                if ok and m[bd.alpha2, q[2], q[1], q[0]]:\n                    nb.append((bd.alpha2, q[2], q[1], q[0]))\n            ze = len(nb)\n            si = S[:, a, z, y, x]\n            tot = sum((S[:, a2, z2, y2, x2] for a2, z2, y2, x2 in nb), np.zeros(B, np.int64))\n            kk = (ze - si * tot) // 2\n            s_idx = (si > 0).astype(np.int64) if plane else np.zeros(B, np.int64)\n            p = ptab[word, s_idx, ze, kk]\n            site = a * L3 * L2 * L1 + z * L2 * L1 + y * L1 + x\n            if independent:\n                u = UL[:B, site]                           # one integer per replica, compared as an integer\n            else:\n                u = U[word, site]\n            S[:, a, z, y, x] = np.where(u < p, -si, si)\n    return S.astype(spins.dtype)\n\n\n# ------------------------------------------------------------------ kernel source\ndef source(sys_: System, independent: bool = False) -> str:\n    """Triton source of the bitwise kernel; same launch structure and class predicate as `generator.source`.\n\n    Grid axis 0 enumerates blocks of G words (G a power of two, a constexpr). One Philox call per site\n    lane and aligned group of four words yields the uniforms of those four words; for G > 4 the G // 4\n    calls of a lane are interleaved back to word order with tl.join.\n\n    independent: one 32-bit uniform per replica, compared bit-sliced with the integer table of `p_fixed`\n    (module docstring); p_ptr then points at that int64 table."""\n    from .generator import _coord_expr, _emit_class, _emit_tile\n\n    J, h, Z = check(sys_)\n    b, L1, L2, L3 = sys_.b, sys_.L1, sys_.L2, sys_.L3\n    D = sys_.crystal.D\n    per = list(sys_.periodic) + [False] * (3 - D)\n    pl = L3 * L2 * L1\n    nsite = b * pl\n    has_mask = sys_.mask is not None\n    NP = max(1, Z.bit_length())\n    Z1 = Z + 1\n    NT = 2 * Z1 * Z1\n    lines: list[str] = []\n    w = lines.append\n    w(\'"""Bitwise multi-spin-coded Ising kernel (generated by Spin_Colour_Kernel.bitwise)."""\')\n    w("import triton")\n    w("import triton.language as tl")\n    w("")\n    w("")\n    w(\'@triton.jit(do_not_specialize=["x_only", "row0", "nword"])\')\n    w("def kernel(src_ptr, dst_ptr, mask_ptr, k1_ptr, p_ptr, state_ptr, x_only, row0, nword,")\n    w("           ALPHA: tl.constexpr, COLOUR: tl.constexpr, BLOCK: tl.constexpr, SYNC: tl.constexpr,")\n    w("           STRIDE: tl.constexpr, BX: tl.constexpr, R: tl.constexpr, G: tl.constexpr):")\n    w("    grp = tl.program_id(0)")\n    w("    cw = tl.arange(0, G)")\n    w("    wv = grp * G + cw")\n    w("    wok = wv < nword")\n    _emit_tile(w, sys_, True)\n    w("    base_seed = tl.load(state_ptr + 0)")\n    w("    sweep = tl.load(state_ptr + 1)")\n    w("    seed = ((base_seed * 1000003 + sweep * 7919 + 12345) % 2147483629).to(tl.int32)")\n    w(f"    rbase = wv.to(tl.int64) * {nsite}")\n    _emit_class(w, sys_)\n    w("    ldw = ld[:, None] & wok[None, :]")\n    w("    s0 = tl.load(src_ptr + rbase[None, :] + own[:, None], mask=ldw, other=0)")\n    w("    ze = tl.zeros([BLOCK], dtype=tl.int32)")\n    for p in range(NP):\n        w(f"    c{p} = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for a in range(b):\n        kw = "if" if a == 0 else "elif"\n        w(f"    {kw} ALPHA == {a}:")\n        bl = bonds(sys_, a)\n        if not bl:\n            w("        pass")\n        for bd in bl:\n            d = list(bd.delta) + [0] * (3 - D)\n            ex, vx = _coord_expr("xc", d[0], L1, per[0])\n            ey, vy = _coord_expr("yv", d[1], L2, per[1])\n            ez, vz = _coord_expr("zv", d[2], L3, per[2])\n            valid = " & ".join(["ld"] + [p for p in (vx, vy, vz) if p])\n            w(f"        q = {bd.alpha2 * pl} + {ez} * {L2 * L1} + {ey} * {L1} + {ex}")\n            w(f"        ok = {valid}")\n            w("        q = tl.where(ok, q, 0)")\n            if has_mask:      # a masked neighbour holds no spin: it is not a neighbour at all\n                w("        ok = ok & (tl.load(mask_ptr + q, mask=ok, other=0) != 0)")\n            w("        ze = ze + ok.to(tl.int32)")\n            w("        tw = tl.load(src_ptr + rbase[None, :] + q[:, None], mask=ok[:, None] & wok[None, :], other=0)")\n            w("        cy = tl.where(ok[:, None], s0 ^ tw, 0)")\n            # ripple adder: plane p receives the carry out of plane p - 1\n            for p in range(NP):\n                if p < NP - 1:\n                    w(f"        t = c{p} & cy")\n                    w(f"        c{p} = c{p} ^ cy")\n                    w("        cy = t")\n                else:\n                    w(f"        c{p} = c{p} ^ cy")\n    if independent:\n        _emit_independent(w, sys_, nsite, NP, Z1, NT, h != 0.0)\n    else:\n        _emit_shared(w, nsite, NP, Z1, NT, h != 0.0)\n    w("    r0 = s0 ^ acc")\n    w("    if SYNC:")\n    wm = "live & (m_own != 0)" if has_mask else "live"\n    w(f"        tl.store(dst_ptr + rbase[None, :] + own[:, None], r0, mask=({wm})[:, None] & wok[None, :])")\n    w("    else:")\n    w("        tl.store(dst_ptr + rbase[None, :] + own[:, None], r0, mask=upd[:, None] & wok[None, :])")\n    return "\\n".join(lines) + "\\n"\n\n\ndef _emit_shared(w, nsite: int, NP: int, Z1: int, NT: int, field: bool) -> None:\n    """Flip mask acc from one uniform per (word, site) shared by the word\'s 32 replicas."""\n    # one Philox call per site lane: output w % 4 of counter (w // 4) * nsite + site\n    w("    if G <= 4:")      # the program\'s words lie in one group of four\n    w(f"        off = (((grp * G) // 4) * {nsite} + own).to(tl.int32)")\n    w("        u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    w("        cq = (wv % 4)[None, :]")\n    w("        u = tl.where(cq == 0, u0[:, None], tl.where(cq == 1, u1[:, None], tl.where(cq == 2, u2[:, None], u3[:, None])))")\n    w("    else:")           # G // 4 whole groups: one call per (site, group), outputs interleaved to word order\n    w(f"        gq = grp * (G // 4) + tl.arange(0, G // 4)")\n    w(f"        off = (gq[None, :] * {nsite} + own[:, None]).to(tl.int32)")\n    w("        u0, u1, u2, u3 = tl.rand4x(seed, off)")\n    w("        u = tl.reshape(tl.join(tl.join(u0, u2), tl.join(u1, u3)), [BLOCK, G])")\n    w(f"    tb = p_ptr + wv[None, :] * {NT} + ze[:, None] * {Z1}")\n    w("    acc = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for k in range(Z1):\n        terms = [f"c{p}" if (k >> p) & 1 else f"~c{p}" for p in range(NP)]\n        w(f"    eq = {\' & \'.join(terms)}")\n        if field:\n            w(f"    pm = tl.load(tb + {k}, mask=ldw, other=0.0)")\n            w(f"    pp = tl.load(tb + {Z1 * Z1 + k}, mask=ldw, other=0.0)")\n            w("    acc = acc | tl.where(u < pp, eq & s0, 0) | tl.where(u < pm, eq & ~s0, 0)")\n        else:\n            w(f"    pk = tl.load(tb + {k}, mask=ldw, other=0.0)")\n            w("    acc = acc | tl.where(u < pk, eq, 0)")\n\n\ndef _emit_independent(w, sys_: System, nsite: int, NP: int, Z1: int, NT: int, field: bool) -> None:\n    """Flip mask acc from one 32-bit uniform per replica, compared bit-sliced with P = floor(p 2^32).\n\n    Table entries (one per k, and per s when field) are selected by lane masks e{i}: the lanes whose count\n    is k (and whose spin is s). Lanes with P = 2^32 accept outright, lanes with 0 < P < 2^32 start undecided,\n    and each level of U, most significant first, decides the lanes whose random bit differs from P\'s bit."""\n    w(f"    tb = p_ptr + wv[None, :] * {NT} + ze[:, None] * {Z1}")\n    J, h, Z = check(sys_)\n    # Metropolis on a periodic, unmasked lattice: every site has z_eff = Z, so the sign of dE of each entry is\n    # known here and the entries with dE <= 0 (p = 1, P = 2^32 in the table) accept without a load or a draw\n    static = sys_.acceptance != "barker" and sys_.mask is None and all(sys_.periodic)\n    w("    acc = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    ent = []           # (lane-mask expression, table offset)\n    for k in range(Z1):\n        terms = [f"c{p}" if (k >> p) & 1 else f"~c{p}" for p in range(NP)]\n        w(f"    eq = {\' & \'.join(terms)}")\n        for sel, off, sigma in ((["eq & ~s0", k, -1], ["eq & s0", Z1 * Z1 + k, 1]) if field else (["eq", k, -1],)):\n            if static and k <= Z and 2.0 * J * (Z - 2 * k) + 2.0 * h * sigma <= 0.0:\n                w(f"    acc = acc | ({sel})")\n                continue\n            w(f"    e{len(ent)} = {sel}")\n            ent.append(off)\n    w("    und = tl.zeros([BLOCK, G], dtype=tl.int32)")\n    for i, off in enumerate(ent):\n        if static:     # z_eff = Z everywhere: one table value per word, shape [1, G]\n            w(f"    P{i} = tl.load(p_ptr + wv[None, :] * {NT} + {Z * Z1 + off}, mask=wok[None, :], other=0)")\n        else:\n            w(f"    P{i} = tl.load(tb + {off}, mask=ldw, other=0)")\n        w(f"    acc = acc | tl.where(P{i} >= {2 ** 32}, e{i}, 0)")\n        w(f"    und = und | tl.where((P{i} > 0) & (P{i} < {2 ** 32}), e{i}, 0)")\n        # undecided lanes have P < 2^32: the level loop works on 32-bit words\n        w(f"    P{i} = P{i}.to(tl.uint32)")\n    # Philox counter word * nsite + site (int32, as in the shared mode), key per level group: grid-independent\n    w(f"    ctr = (rbase[None, :] + own[:, None].to(tl.int64)).to(tl.int32)")\n\n    def group(indent: str, gexpr: str):\n        w(f"{indent}kg = ((seed.to(tl.int64) + ({gexpr} + 1) * {GROUP_KEY_STEP}) % 2147483629).to(tl.int32)")\n        w(f"{indent}q0, q1, q2, q3 = tl.randint4x(kg, ctr, n_rounds={INDEPENDENT_ROUNDS})")\n        for q in range(4):\n            w(f"{indent}rw = q{q}.to(tl.int32, bitcast=True)")\n            w(f"{indent}sh = ({LEVELS - 1 - q} - 4 * {gexpr}).to(tl.uint32)")\n            # bit sh of each entry\'s P as an all-ones / all-zeros mask, then one AND per entry\n            masks = " | ".join(f"(e{i} & (0 - ((P{i} >> sh) & 1)).to(tl.int32, bitcast=True))" for i in range(len(ent))) or "0"\n            w(f"{indent}pb = {masks}")\n            w(f"{indent}acc = acc | (und & ~rw & pb)")\n            w(f"{indent}und = und & ~(rw ^ pb)")\n\n    # the first two level groups (8 levels) run unconditionally: they decide all but about 2^-8 of the lanes,\n    # and the tile-wide reduction is paid only once, for the rare tail; the stream and the result are unchanged\n    if ent:\n        group("    ", "(tl.program_id(0) * 0).to(tl.int64)")\n        group("    ", "(tl.program_id(0) * 0 + 1).to(tl.int64)")\n    w("    g = tl.program_id(0) * 0 + 2")\n    w("    nund = tl.sum(tl.sum((und != 0).to(tl.int32), axis=1), axis=0)")\n    w(f"    while (g < {LEVELS // 4}) & (nund > 0):")\n    group("        ", "g.to(tl.int64)")\n    w("        g = g + 1")\n    w("        nund = tl.sum(tl.sum((und != 0).to(tl.int32), axis=1), axis=0)")\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "a4488d0ada766a05b1381a32f9744a867849e8547d40e8d9c6c03d36822f225d", "bitwise.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/bitwise.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/bitwise.py  sha256 a4488d0ada766a05…")

In [ ]:
# Spin_Colour_Kernel/sampler.py — byte-identical to the repository. sha256 86a9893be5ac75b74ca1e1a75acf15aadb336e4c5f1514453f6e32977fd3c52b
import hashlib, os
_src = '"""Launch plumbing for a generated kernel: sweeps, the sequential reference, observables.\n\nA parallel sweep is n_c * b launches (colour-major, then sublattice). The sequential reference\nvisits the same classes in the same order and, inside each class, the sites one at a time in\nraster order (z, y, x), through the same compiled kernel. Because no two sites of one class are\nbonded (7), the two chains must hold identical spins after every sweep (hypothesis H1).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\n\nfrom . import bitwise as bw\nfrom .generator import compile_system, x_period\nfrom .system import System\n\n\n#: smallest x-period worth compacting; E6 on a T4 (2026-11-07) lost 25-35% with stride-2 loads\nCOMPACT_MIN = 4\n#: lanes a compacted program aims for: R rows of BX sites each\nTILE = 128\n#: float kernels: lanes per one-row program. A longer row is split over grid axis 2. Above 1024 lanes a 4-warp\n#: Heisenberg program spills its registers to local memory (Nsight on the H100: 0, 18 M and 253 M local sectors at\n#: rows of 1024, 2048 and 4096 sites), so rows up to 1024 sites keep one program, longer rows are chunked.\nMAX_BLOCK = 1024\n#: bit-packed Ising: lanes per program (each lane carries a 32-replica tile, so BLOCK x 32 values)\nPTILE = 64\n#: replicas per packed word\nWORD = 32\n#: bitwise: default words per program (one Philox call per site serves an aligned group of four words)\nBWG = 4\n\n\ndef _num_warps(block: int) -> int:\n    return 4 if block >= 256 else (2 if block >= 128 else 1)\n\n\nclass Sampler:\n    def __init__(self, system: System, batch: int, device="cuda", seed: int = 0, compact: bool = True,\n                 spin_dtype: torch.dtype | None = None, packed: bool | None = None, bitwise: bool = False,\n                 words_per_program: int | None = None, independent: bool = False, bitwise_tile: int | None = None,\n                 bitwise_warps: int | None = None, max_block: int = MAX_BLOCK):\n        """bitwise: true multi-spin coding (module `bitwise`): packed storage updated with bit operations, one\n        uniform per (word, site, sweep) shared by the word\'s 32 replicas, one temperature per word.\n        independent (bitwise only): every replica draws its own 32-bit uniform U and flips iff U < floor(p 2^32),\n        compared bit-sliced; replicas of a word no longer share randomness (p quantized down to 2^-32).\n        words_per_program: bitwise words a program covers (power of two, default BWG; fewer when the batch is small)."""\n        self.sys = system\n        self.bitwise = bool(bitwise)\n        self.independent = bool(independent)\n        if self.independent and not self.bitwise:\n            raise ValueError("independent per-replica draws need bitwise=True")\n        if self.bitwise:\n            _, _, self.zmax = bw.check(system)     # ValueError for systems the bitwise kernel cannot run\n            if packed is False:\n                raise ValueError("bitwise multi-spin coding uses the packed storage")\n            packed = True\n        # Ising defaults to multi-spin coding (32 replicas per int32 word, see generator.source) once the\n        # batch fills a word; below that a word computes 32 replicas for fewer than 32 used, and int8 is faster\n        self.packed = (system.n == 1 and int(batch) >= WORD) if packed is None else bool(packed)\n        if self.packed and system.n != 1:\n            raise ValueError("bit packing is for Ising (n = 1) only")\n        # Ising spins are +-1 (0 on masked sites): int8 storage reads a quarter of the bytes of float32 and holds\n        # them exactly; vector models keep float32 components\n        self.spin_dtype = spin_dtype or (torch.int8 if system.n == 1 else torch.float32)\n        self.B = int(batch)\n        self.device = torch.device(device)\n        self.kernel, self.src = compile_system(system, self.packed, self.bitwise, self.independent)\n        n, b, L3, L2, L1 = system.n, system.b, system.L3, system.L2, system.L1\n        self.W = -(-self.B // WORD) if self.packed else self.B      # program rows of grid axis 0\n        self.Bp = self.W * WORD if self.packed else self.B           # replicas the kernel computes\n        if self.Bp * b * L3 * L2 * L1 >= 2 ** 31 - 1:\n            raise ValueError("replica batch too large for the int32 Philox counter")\n        # bitwise: a program covers G words, a power of two; G >= 4 wastes no Philox output\n        gmax = words_per_program or BWG\n        if gmax & (gmax - 1):\n            raise ValueError("words_per_program must be a power of two")\n        self.G = min(gmax, 1 << (self.W - 1).bit_length()) if self.bitwise else 1\n        self.ngroups = -(-self.W // self.G)\n        self.nrows = L3 * L2\n        if max_block < 16 or max_block & (max_block - 1):\n            raise ValueError("max_block must be a power of two >= 16")\n        self.block = min(max(16, 1 << (L1 - 1).bit_length()), max_block)\n        self.chunks = -(-L1 // self.block)            # one-row programs over grid axis 2 (rows longer than max_block)\n        self.nw = _num_warps(self.block)\n        # colour compaction: a program covers R rows x BX sites of one colour (BX = ceil(L1 / P))\n        P = x_period(system.colouring) if compact else 1\n        P = P if P >= COMPACT_MIN else 1       # below it, strided loads cost more than the masked lanes save\n        self.stride = P\n        self.bx = -(-L1 // P)                         # sites of one colour per row\n        tile_c = bitwise_tile if (bitwise and bitwise_tile) else TILE   # bitwise: lanes per compacted program\n        self.rows_per = max(1, min(self.nrows, tile_c // self.bx)) if P > 1 else 1\n        if self.packed and not self.bitwise:   # a bitwise lane holds G int32 words, not a 32-float tile\n            self.rows_per = max(1, min(self.nrows, PTILE // self.bx)) if P > 1 else 1\n        self.cblock = max(16, 1 << (self.rows_per * self.bx - 1).bit_length())\n        self.cchunks = 1\n        if not self.packed and self.rows_per == 1 and self.cblock > max_block:\n            self.cblock = max_block\n            self.cchunks = -(-self.bx // max_block)\n        self.cnw = _num_warps(self.cblock)\n        if self.packed:   # one-row programs cover at most PTILE sites of the row; longer rows go over grid axis 2\n            nx = self.bx if P > 1 else L1\n            tile = TILE if self.bitwise else PTILE\n            self.pblock = min(tile, max(16, 1 << (nx - 1).bit_length()))\n            self.pchunks = -(-nx // self.pblock)\n            self.pblock_ref = min(tile, self.block)\n            self.pchunks_ref = -(-L1 // self.pblock_ref)\n            if not self.bitwise:\n                self.cnw = 4 if self.cblock * WORD >= 2048 else 2\n            elif bitwise_tile:\n                self.pblock = min(bitwise_tile, max(16, 1 << (nx - 1).bit_length()))\n                self.pchunks = -(-nx // self.pblock)\n            if self.bitwise and bitwise_warps:     # tuning knob of the bitwise kernels (E6 pilot)\n                self.cnw = self.nw = int(bitwise_warps)\n        dev = self.device\n        m = system.site_mask()\n        self.mask_np = m\n        self.mask = torch.as_tensor(m.astype(np.int8), device=dev)\n        k1 = system.cubic_k1 if system.cubic_k1 is not None else np.zeros(m.shape, np.float32)\n        self.k1 = torch.as_tensor(np.asarray(k1, np.float32), device=dev)\n        self.state = torch.tensor([int(seed), 0], dtype=torch.int64, device=dev)\n        self.beta = torch.ones(self.Bp, dtype=torch.float32, device=dev)\n        if self.bitwise:     # acceptance table p[word, s, z_eff, k], rebuilt in place by set_temperature\n            self._table = bw.p_fixed if self.independent else bw.p_table     # int64 P = floor(p 2^32), or float32 p\n            self.ptab = torch.as_tensor(self._table(system, np.ones(self.W)), device=dev)\n        self.colours = system.colour_map()\n        self.classes = [(c, a) for c in range(system.colouring.n_colours) for a in range(b)\n                        if np.any((self.colours[a] == c) & m[a])]\n        mt = torch.as_tensor(m, device=dev)\n        self._mask_f = mt.to(torch.float32)\n\n    # ------------------------------------------------------------ state\n    def set_temperature(self, T):\n        T = torch.as_tensor(T, dtype=torch.float32, device=self.device).reshape(-1)\n        beta = (1.0 / T).expand(self.B).contiguous() if T.numel() == 1 else (1.0 / T).contiguous()\n        if self.Bp > self.B:     # padding replicas of the last word: computed, never observed\n            beta = torch.cat([beta, beta[-1:].expand(self.Bp - self.B)])\n        if self.bitwise:\n            bwd = beta.reshape(self.W, WORD)\n            if not torch.all(bwd == bwd[:, :1]):\n                raise ValueError("bitwise multi-spin coding needs one temperature per word of 32 replicas")\n            # in place, like beta: a captured CUDA graph reads the table by address\n            self.ptab.copy_(torch.as_tensor(self._table(self.sys, bwd[:, 0].double().cpu().numpy()), device=self.device))\n        # in place: a CUDA graph captured earlier reads this buffer by address, so a new tensor would be ignored\n        self.beta.copy_(beta)\n\n    def set_seed(self, seed: int, sweep: int = 0):\n        self.state[0] = int(seed)\n        self.state[1] = int(sweep)\n\n    def init_spins(self, kind="random", seed: int = 0) -> torch.Tensor:\n        s = self.sys\n        g = torch.Generator(device="cpu").manual_seed(int(seed))\n        shape = (self.B, s.n) + s.shape\n        if kind == "ordered":\n            x = torch.zeros(shape)\n            x[:, 0] = 1.0\n        elif s.n == 1:\n            x = torch.randint(0, 2, shape, generator=g).float() * 2 - 1\n        else:\n            x = torch.randn(shape, generator=g)\n            x = x / x.norm(dim=1, keepdim=True)\n        x = x.to(self.device) * self._mask_f\n        if self.packed:\n            return self.pack(x)\n        return x.to(self.spin_dtype).contiguous()\n\n    def pack(self, x: torch.Tensor) -> torch.Tensor:\n        """(B, 1, b, L3, L2, L1) spins of +-1 (0 on masked sites) -> (W, 1, b, L3, L2, L1) int32 words."""\n        bits = (x > 0).to(torch.int64)\n        if self.Bp > self.B:\n            bits = torch.cat([bits, bits.new_zeros((self.Bp - self.B,) + tuple(bits.shape[1:]))])\n        bits = bits.reshape((self.W, WORD) + tuple(bits.shape[1:]))\n        sh = (torch.arange(WORD, device=bits.device, dtype=torch.int64)).reshape((1, WORD) + (1,) * (bits.dim() - 2))\n        word = (bits << sh).sum(dim=1)\n        word = torch.where(word >= 2 ** 31, word - 2 ** 32, word)\n        return word.to(torch.int32).contiguous()\n\n    def unpack(self, S: torch.Tensor) -> torch.Tensor:\n        """Packed words -> (B, 1, b, L3, L2, L1) int8 spins of +-1, 0 on masked sites; identity when unpacked."""\n        if not self.packed or S.dtype != torch.int32:     # already spins\n            return S\n        w = S.to(torch.int64) & 0xFFFFFFFF\n        sh = torch.arange(WORD, device=S.device, dtype=torch.int64).reshape((1, WORD) + (1,) * (S.dim() - 1))\n        bits = (w.unsqueeze(1) >> sh) & 1\n        x = (bits * 2 - 1).reshape((self.Bp,) + tuple(S.shape[1:]))[: self.B]\n        return (x * self._mask_f.to(torch.int64)).to(torch.int8)\n\n    # ------------------------------------------------------------ sweeps\n    def _launch(self, S, dst, c, a, grid_rows, row0=0, x_only=-1, sync=False, compact=False):\n        if self.bitwise:\n            if compact and self.rows_per > 1:\n                grid = (self.ngroups, -(-grid_rows // self.rows_per), 1)\n                kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n            elif compact:\n                grid = (self.ngroups, grid_rows, self.pchunks)\n                kw = dict(BLOCK=self.pblock, STRIDE=self.stride, BX=self.pblock, R=1, num_warps=self.nw)\n            else:\n                grid = (self.ngroups, grid_rows, self.pchunks_ref)\n                kw = dict(BLOCK=self.pblock_ref, STRIDE=1, BX=self.pblock_ref, R=1, num_warps=self.nw)\n            args = (S, dst, self.mask, self.k1, self.ptab, self.state, x_only, row0, self.W)\n            kw["G"] = self.G\n            if self.device.type == "cuda":\n                with torch.cuda.device(self.device):\n                    self.kernel[grid](*args, ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n                return\n            self.kernel[grid](*args, ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n            return\n        if self.packed:\n            if compact and self.rows_per > 1:\n                grid = (self.W, -(-grid_rows // self.rows_per), 1)\n                kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n            elif compact:\n                grid = (self.W, grid_rows, self.pchunks)\n                kw = dict(BLOCK=self.pblock, STRIDE=self.stride, BX=self.pblock, R=1, num_warps=4)\n            else:\n                grid = (self.W, grid_rows, self.pchunks_ref)\n                kw = dict(BLOCK=self.pblock_ref, STRIDE=1, BX=self.pblock_ref, R=1, num_warps=4)\n        elif compact:\n            grid = (self.B, -(-grid_rows // self.rows_per), self.cchunks)\n            kw = dict(BLOCK=self.cblock, STRIDE=self.stride, BX=self.bx, R=self.rows_per, num_warps=self.cnw)\n        else:\n            grid = (self.B, grid_rows, self.chunks)\n            kw = dict(BLOCK=self.block, STRIDE=1, BX=self.block, R=1, num_warps=self.nw)\n        if self.device.type == "cuda":\n            with torch.cuda.device(self.device):\n                self.kernel[grid](S, dst, self.mask, self.k1, self.beta, self.state, x_only, row0,\n                                  ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n            return\n        self.kernel[grid](S, dst, self.mask, self.k1, self.beta, self.state, x_only, row0,\n                          ALPHA=a, COLOUR=c, SYNC=sync, **kw)\n\n    def sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """n coloured sweeps in place; advances the sweep counter."""\n        for _ in range(n):\n            for c, a in self.classes:\n                self._launch(S, S, c, a, self.nrows, compact=self.stride > 1)\n            self.state[1] += 1\n        return S\n\n    def graph(self, S: torch.Tensor, k: int):\n        """Capture k sweeps (launches and counter increments) in a CUDA graph; returns its replay.\n\n        Removes the host launch cost from timings; the counter stream is the same as k calls of\n        `sweep`, because the sweep counter is incremented on the device inside the graph.\n        """\n        with torch.cuda.device(self.device):\n            return self._graph(S, k)\n\n    def _graph(self, S: torch.Tensor, k: int):\n        s = torch.cuda.Stream(device=self.device)\n        s.wait_stream(torch.cuda.current_stream(self.device))\n        with torch.cuda.stream(s):\n            self.sweep(S, 1)          # warm-up compiles every class launch outside the capture\n        torch.cuda.current_stream(self.device).wait_stream(s)\n        torch.cuda.synchronize(self.device)\n        g = torch.cuda.CUDAGraph()\n        with torch.cuda.graph(g):\n            self.sweep(S, k)\n        dev = self.device\n\n        def replay():\n            with torch.cuda.device(dev):\n                g.replay()\n        # the graph addresses this sampler\'s buffers and S directly: the replay keeps them alive, or a caller\n        # that drops the sampler (or rebinds S) before replaying reads freed memory\n        replay.keep = (self, S, g)\n        return replay\n\n    def sync_sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """The negative control of E2: every site updated in the same step from the old state."""\n        for _ in range(n):\n            dst = S.clone()\n            for a in range(self.sys.b):\n                self._launch(S, dst, -1, a, self.nrows, sync=True)\n            S.copy_(dst)\n            self.state[1] += 1\n        return S\n\n    def reference_sweep(self, S: torch.Tensor, n: int = 1) -> torch.Tensor:\n        """Sequential reference chain: one launch per site, same kernel, same counters."""\n        L2, L1 = self.sys.L2, self.sys.L1\n        order = []\n        for c, a in self.classes:\n            sites = np.argwhere((self.colours[a] == c) & self.mask_np[a])   # (z, y, x), raster order\n            order.append((c, a, sites))\n        for _ in range(n):\n            for c, a, sites in order:\n                for z, y, x in sites:\n                    self._launch(S, S, c, a, 1, row0=int(z) * L2 + int(y), x_only=int(x))\n            self.state[1] += 1\n        return S\n\n    # ------------------------------------------------------------ observables\n    def local_field(self, S: torch.Tensor) -> torch.Tensor:\n        """Bond part of the local field (3) for every site, by rolls (float64), as a reference."""\n        s = self.sys\n        D = s.crystal.D\n        S = self.unpack(S)\n        S64 = S.double()\n        v = torch.zeros_like(S64)\n        dims = {0: 5, 1: 4, 2: 3}       # lattice axis k -> tensor dim\n        for i, bd in enumerate(s.crystal.bonds):\n            M = torch.as_tensor(s.bond_matrix(i), dtype=torch.float64, device=S.device)\n            if not M.any():\n                continue\n            src = S64[:, :, bd.alpha2]\n            valid = torch.ones(s.shape[1:], dtype=torch.float64, device=S.device)\n            for k in range(D):\n                d = bd.delta[k]\n                if d == 0:\n                    continue\n                src = torch.roll(src, shifts=-d, dims=dims[k] - 1)\n                if not s.periodic[k]:\n                    Lk = s.box[k]\n                    idx = torch.arange(Lk, device=S.device)\n                    ok = ((idx + d) >= 0) & ((idx + d) < Lk)\n                    shp = [1, 1, 1]\n                    shp[2 - k] = Lk\n                    valid = valid * ok.to(torch.float64).reshape(shp)\n            src = src * valid\n            v[:, :, bd.alpha] += torch.einsum("kl,blzyx->bkzyx", M, src)\n        return v\n\n    def energy(self, S: torch.Tensor) -> torch.Tensor:\n        """Total energy per replica, eq. (1), float64."""\n        s = self.sys\n        S = self.unpack(S)\n        S64 = S.double()\n        v = self.local_field(S)\n        E = -0.5 * (S64 * v).sum(dim=(1, 2, 3, 4, 5))\n        if s.h is not None:\n            h = torch.as_tensor(s.h, dtype=torch.float64, device=S.device).reshape(1, -1, 1, 1, 1, 1)\n            E = E - (S64 * h).sum(dim=(1, 2, 3, 4, 5))\n        if s.cubic_k1 is not None:\n            x, y, z = S64[:, 0], S64[:, 1], S64[:, 2]\n            k1 = self.k1.double()\n            a = k1 * ((x * y) ** 2 + (x * z) ** 2 + (y * z) ** 2) + s.cubic_k2 * (x * y * z) ** 2\n            E = E + a.sum(dim=(1, 2, 3, 4))\n        return E\n\n    def magnetization(self, S: torch.Tensor) -> torch.Tensor:\n        """Per-sublattice magnetization M_alpha, shape (B, b, n), float64."""\n        return self.unpack(S).double().sum(dim=(3, 4, 5)).transpose(1, 2)\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "86a9893be5ac75b74ca1e1a75acf15aadb336e4c5f1514453f6e32977fd3c52b", "sampler.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/sampler.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/sampler.py  sha256 86a9893be5ac75b7…")

In [ ]:
# Spin_Colour_Kernel/simulate.py — byte-identical to the repository. sha256 d97c96ad76f4aa4d9b6cf261222da69996f9683e4f18e77a42bf12cc4e2fed8a
import hashlib, os
_src = '"""General simulation of a user-defined spin Hamiltonian, behind notebooks/spin_simulation.ipynb.\n\nA run is one model (Ising, XY or Heisenberg), one crystal, one box, one set of active sites and a list\nof states; a state is one parameter set theta of the Hamiltonian\n\n    H = - sum_<ij> J_s  S_i . S_j                       exchange, per neighbour shell s\n        - sum_<ij> S_i^T A_ij S_j                       interfacial DMI, A_lm = D sum_k r_k eps_klm\n                                                        (in-plane first-shell bonds, Heisenberg only)\n        + sum_i K1_i (sx^2 sy^2 + sx^2 sz^2 + sy^2 sz^2) + K2 sum_i (sx sy sz)^2     cubic anisotropy\n        - sum_i H . S_i                                 Zeeman field\n\nwith K1_i = Ks on surface sites (fewer active first-shell neighbours than the bulk). Every replica of a\nstate follows the same temperature ladder, from T_max down to T_min in steps of T_step; at each\ntemperature it thermalizes, then measures every `measure_every` sweeps.\n\ntheta keys: J {shell: value}, D, K1, Ks, K2, H, H_theta, H_phi. The field direction is\n(sin t cos p, sin t sin p, cos t) for Heisenberg (t = H_theta, p = H_phi, degrees; t = 0 is +z),\n(cos p, sin p) for XY and +1 for Ising.\n\nOutputs, in the run directory:\n    run.json             the configuration and the derived ladder, sites and units\n    sites.npy            (n_sites, 3) Cartesian positions of the active sites\n    state_XXX.npz        per state: M (B, nT, 3), E, E2, Mabs, M2 (B, nT)\n    configurations.npy   (n_states, B, n_saved, n_sites, 3) float32 spins every config_every T steps\n    magnetization.npy    (n_states, B, nT, 3), written by `collect`\n    results.npz          every combined array, Cv and chi included, written by `collect`\n\nRun as a module to process some states of a prepared run on one device:\n    python -m Spin_Colour_Kernel.simulate --run DIR --states 0,2 --device cuda\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom . import crystals as cr\nfrom .sampler import Sampler\nfrom .system import System\n\nKB_MEV = 0.086173404          # meV / K\nMODELS = {"ising": 1, "xy": 2, "heisenberg": 3}\nCOORDS = {"cartesian": ("x", "y", "z"), "spherical": ("r", "theta", "phi"),\n          "cylindrical": ("rho", "phi", "z")}\n_MATH = {k: getattr(np, k) for k in ("sqrt", "sin", "cos", "tan", "arctan2", "arccos", "abs", "exp",\n                                     "log", "minimum", "maximum", "pi", "mod", "floor", "ceil")}\n_MATH["abs"] = np.abs\n\n\n# ------------------------------------------------------------------ geometry\ndef _positions(crystal: cr.Crystal, box) -> np.ndarray:\n    """Cartesian position of every site, (b, L3, L2, L1, 3), centred on the box."""\n    D = crystal.D\n    L1, L2 = int(box[0]), int(box[1])\n    L3 = int(box[2]) if D == 3 else 1\n    z, y, x = np.meshgrid(np.arange(L3), np.arange(L2), np.arange(L1), indexing="ij")\n    cells = np.stack([x, y, z][:D], axis=-1).astype(float)          # (L3, L2, L1, D)\n    out = np.zeros((crystal.b, L3, L2, L1, 3))\n    for a in range(crystal.b):\n        out[a, ..., :D] = (cells + crystal.basis[a]) @ crystal.vectors\n    return out - out.reshape(-1, 3).mean(axis=0)\n\n\ndef active_sites(crystal, box, coords: str = "", shape: str = ""):\n    """(positions (b, L3, L2, L1, 3), mask (b, L3, L2, L1)) for a shape written in a coordinate system.\n\n    coords is "cartesian" (x, y, z), "spherical" (r, theta, phi) or "cylindrical" (rho, phi, z), with the\n    origin at the centre of the box and angles in radians; shape is a condition in those names, for\n    example "r <= 10" or "(rho <= 8) & (abs(z) <= 2)". Both empty: every site is active.\n    """\n    c = crystal if isinstance(crystal, cr.Crystal) else cr.make(str(crystal))\n    if len(box) != c.D:\n        raise ValueError(f"crystal {c.name} needs a box of {c.D} sizes, got {list(box)}")\n    pos = _positions(c, box)\n    coords, shape = (coords or "").strip().lower(), (shape or "").strip()\n    if not shape:\n        if coords and coords not in COORDS:\n            raise ValueError(f"unknown coordinate system {coords!r}; known: {\', \'.join(COORDS)}")\n        return pos, np.ones(pos.shape[:-1], bool)\n    if not coords:\n        raise ValueError("a shape needs its coordinate system: set coords to one of " + ", ".join(COORDS))\n    if coords not in COORDS:\n        raise ValueError(f"unknown coordinate system {coords!r}; known: {\', \'.join(COORDS)}")\n    x, y, z = pos[..., 0], pos[..., 1], pos[..., 2]\n    rho = np.hypot(x, y)\n    r = np.sqrt(x * x + y * y + z * z)\n    phi = np.arctan2(y, x)\n    names = {"cartesian": dict(x=x, y=y, z=z),\n             "spherical": dict(r=r, theta=np.arccos(np.divide(z, r, out=np.ones_like(z), where=r > 0)), phi=phi),\n             "cylindrical": dict(rho=rho, phi=phi, z=z)}[coords]\n    try:\n        mask = eval(shape, {"__builtins__": {}}, {**_MATH, **names})   # the user\'s own notebook cell\n    except NameError as e:\n        raise ValueError(f"{e}; {coords} coordinates are {\', \'.join(COORDS[coords])}") from None\n    mask = np.broadcast_to(np.asarray(mask, bool), pos.shape[:-1]).copy()\n    if not mask.any():\n        raise ValueError(f"the shape {shape!r} contains no site of the box")\n    return pos, mask\n\n\n# ------------------------------------------------------------------ Hamiltonian\ndef _f(theta, key, default=0.0) -> float:\n    v = theta.get(key, default)\n    return float(default if v is None else v)\n\n\ndef _dmi(crystal: cr.Crystal, D: float) -> dict[int, np.ndarray]:\n    """A_lm = D sum_k r_k eps_klm on in-plane first-shell bonds (r the unit bond vector)."""\n    eps = np.zeros((3, 3, 3))\n    for i, j, k in ((0, 1, 2), (1, 2, 0), (2, 0, 1)):\n        eps[i, j, k], eps[i, k, j] = 1.0, -1.0\n    out = {}\n    for i, bd in enumerate(crystal.bonds):\n        if bd.shell != 1:\n            continue\n        r = np.zeros(3)\n        r[:crystal.D] = (np.asarray(bd.delta, float) + crystal.basis[bd.alpha2] - crystal.basis[bd.alpha]) @ crystal.vectors\n        if abs(r[2]) > 1e-9:\n            continue\n        r /= np.linalg.norm(r)\n        out[i] = D * np.einsum("k,klm->lm", r, eps)\n    return out\n\n\ndef _surface(crystal: cr.Crystal, box, periodic, mask: np.ndarray) -> np.ndarray:\n    """Active sites with fewer active first-shell neighbours than a bulk site of their basis index."""\n    full = {a: 0 for a in range(crystal.b)}\n    count = np.zeros(mask.shape, int)\n    axes = {0: 3, 1: 2, 2: 1}                      # lattice axis -> array axis of (b, L3, L2, L1)\n    for bd in crystal.bonds:\n        if bd.shell != 1:\n            continue\n        full[bd.alpha] += 1\n        src = mask[bd.alpha2].astype(int)\n        valid = np.ones(mask.shape[1:], int)\n        for k in range(crystal.D):\n            d = bd.delta[k]\n            if d == 0:\n                continue\n            src = np.roll(src, -d, axis=axes[k] - 1)\n            if not periodic[k]:\n                idx = np.arange(box[k])\n                ok = ((idx + d) >= 0) & ((idx + d) < box[k])\n                shp = [1, 1, 1]\n                shp[2 - k] = box[k]\n                valid = valid * ok.astype(int).reshape(shp)\n        count[bd.alpha] += src * valid\n    bulk = np.stack([np.full(mask.shape[1:], full[a]) for a in range(crystal.b)])\n    return mask & (count < bulk)\n\n\ndef build_system(model: str, crystal: str, box, periodic, theta: dict, mask=None, log=None) -> System:\n    """The System of one state theta. A periodic box with no admissible colouring falls back to open."""\n    model = str(model).strip().lower()\n    if model not in MODELS:\n        raise ValueError(f"unknown model {model!r}; known: {\', \'.join(MODELS)}")\n    n = MODELS[model]\n    J = {int(k): float(v) for k, v in (theta.get("J") or {1: 1.0}).items()}\n    D, K1, K2, H = _f(theta, "D"), _f(theta, "K1"), _f(theta, "K2"), _f(theta, "H")\n    Ks = _f(theta, "Ks", K1)\n    if n != 3 and any(_f(theta, k) for k in ("D", "K1", "Ks", "K2")):\n        raise ValueError(f"D, K1, Ks and K2 need the Heisenberg model; {model} carries J and H only")\n    unknown = set(theta) - {"name", "J", "D", "K1", "Ks", "K2", "H", "H_theta", "H_phi"}\n    if unknown:\n        raise ValueError(f"unknown parameters {sorted(unknown)}; known: J, D, K1, Ks, K2, H, H_theta, H_phi")\n    shells = tuple(sorted({s for s, v in J.items() if v != 0.0} | {1}))\n    c = cr.make(str(crystal), shells)\n    box = tuple(int(v) for v in box)\n    periodic = tuple(bool(v) for v in periodic)\n    m = None if mask is None else np.asarray(mask, bool)\n    if m is not None and m.shape != (c.b, box[2] if c.D == 3 else 1, box[1], box[0]):\n        raise ValueError(f"mask shape {m.shape} does not match the box {box} of {c.name}")\n    t, p = np.deg2rad(_f(theta, "H_theta")), np.deg2rad(_f(theta, "H_phi"))\n    h = None\n    if H:\n        h = {1: (H,), 2: (H * np.cos(p), H * np.sin(p)),\n             3: (H * np.sin(t) * np.cos(p), H * np.sin(t) * np.sin(p), H * np.cos(t))}[n]\n        h = tuple(float(v) for v in h)\n    kw = dict(extra=_dmi(c, D) if (n == 3 and D) else {}, h=h, cubic_k2=K2, mask=m)\n    name = theta.get("name") or f"{model} {c.name}"\n\n    def make(per):\n        k1 = None\n        if n == 3 and (K1 or Ks or K2):\n            mm = np.ones((c.b, box[2] if c.D == 3 else 1, box[1], box[0]), bool) if m is None else m\n            k1 = (np.where(_surface(c, box, per, mm), Ks, K1) * mm).astype(np.float32)\n        return System(str(name), n, c, box, per, J, cubic_k1=k1, **kw)\n    try:\n        return make(periodic)\n    except ValueError as e:\n        if not any(periodic) or "colouring" not in str(e):\n            raise\n        (log or print)(f"  no admissible colouring for the periodic box {box}: using open boundaries")\n        return make((False,) * c.D)\n\n\ndef temperature_ladder(T_max: float, T_min: float, T_step: float) -> np.ndarray:\n    """Descending temperatures T_max, T_max - T_step, ..., T_min."""\n    T_max, T_min, T_step = float(T_max), float(T_min), float(T_step)\n    if T_step <= 0 or T_min <= 0 or T_max < T_min:\n        raise ValueError("need T_max >= T_min > 0 and T_step > 0")\n    k = int(np.floor((T_max - T_min) / T_step + 1e-9))\n    return np.round(T_max - T_step * np.arange(k + 1), 10)\n\n\ndef size_estimate_gb(n_states: int, replicas: int, n_saved: int, n_sites: int) -> float:\n    """Size of configurations.npy (float32, three components)."""\n    return n_states * replicas * n_saved * n_sites * 3 * 4 / 1e9\n\n\n# ------------------------------------------------------------------ one state\ndef _spins(A: Sampler, S: torch.Tensor, mask: np.ndarray) -> np.ndarray:\n    """(B, n_active, 3) float32 spins of the active sites, zero third (and second) components padded."""\n    x = A.unpack(S).float().permute(0, 2, 3, 4, 5, 1).cpu().numpy()   # (B, b, L3, L2, L1, n)\n    x = x[:, mask]\n    out = np.zeros(x.shape[:2] + (3,), np.float32)\n    out[..., :x.shape[-1]] = x\n    return out\n\n\ndef simulate_state(s: System, replicas: int, temps, n_therm: int, n_meas: int, every: int, config_every: int,\n                   seed: int, device="cpu", kB: float = 1.0, log=print) -> dict:\n    """Run the ladder for one state. Energies are totals; M is the mean magnetization per site."""\n    A = Sampler(s, int(replicas), device, seed)\n    S = A.init_spins(seed=seed)\n    mask = s.site_mask()\n    n_act = float(s.n_active)\n    k = max(1, int(every))\n    m = max(1, int(n_meas) // k)\n    ce = max(1, int(config_every))\n    temps = np.asarray(temps, float)\n    nT, B = len(temps), int(replicas)\n    saved = [t for t in range(nT) if t % ce == 0]\n    out = dict(M=np.zeros((B, nT, 3), np.float32), E=np.zeros((B, nT)), E2=np.zeros((B, nT)),\n               Mabs=np.zeros((B, nT)), M2=np.zeros((B, nT)),\n               configs=np.zeros((B, len(saved), int(mask.sum()), 3), np.float32),\n               config_temperatures=temps[saved])\n\n    def observe():\n        E = A.energy(S)\n        Mv = A.magnetization(S).sum(dim=1) / n_act               # (B, n)\n        return E, Mv\n\n    t0 = time.perf_counter()\n    for t, T in enumerate(temps):\n        A.set_temperature(kB * float(T))\n        if n_therm > 0:\n            A.sweep(S, int(n_therm))\n        E_ref = observe()[0]\n        dE = torch.zeros_like(E_ref)\n        dE2, Ma, M2 = torch.zeros_like(E_ref), torch.zeros_like(E_ref), torch.zeros_like(E_ref)\n        Mv_acc = None\n        for _ in range(m):\n            A.sweep(S, k)\n            E, Mv = observe()\n            d = E - E_ref\n            dE += d\n            dE2 += d * d\n            a = Mv.norm(dim=1)\n            Ma += a\n            M2 += a * a\n            Mv_acc = Mv.clone() if Mv_acc is None else Mv_acc + Mv\n        dE, dE2 = dE / m, dE2 / m\n        out["E"][:, t] = (E_ref + dE).cpu().numpy()\n        out["E2"][:, t] = (dE2 + 2 * E_ref * dE + E_ref * E_ref).cpu().numpy()\n        out["Mabs"][:, t] = (Ma / m).cpu().numpy()\n        out["M2"][:, t] = (M2 / m).cpu().numpy()\n        out["M"][:, t, :s.n] = (Mv_acc / m).float().cpu().numpy()\n        if t in saved:\n            out["configs"][:, saved.index(t)] = _spins(A, S, mask)\n        if t % max(1, nT // 10) == 0 or t == nT - 1:\n            log(f"    T = {T:9.4f}   <E>/site = {out[\'E\'][:, t].mean() / n_act:+.4f}"\n                f"   <|M|> = {out[\'Mabs\'][:, t].mean():.4f}   {time.perf_counter() - t0:8.1f} s")\n    return out\n\n\n# ------------------------------------------------------------------ a run\ndef default_config() -> dict:\n    """Every setting of a run, with working defaults (a small sc Heisenberg ferromagnet)."""\n    return dict(\n        model="heisenberg",               # ising | xy | heisenberg\n        crystal="sc",                     # square triangular honeycomb kagome (2D) | sc bcc fcc (3D)\n        box=[12, 12, 12],                 # cells along x, y[, z]\n        periodic=[True, True, True],\n        coords="",                        # "" | cartesian | spherical | cylindrical\n        shape="",                         # condition in those coordinates; "" = every site active\n        states=[{"name": "J=1", "J": {"1": 1.0}}],\n        units="reduced",                  # reduced: J and T in units of J (kB = 1) | meV: T in kelvin\n        T_max=2.0, T_min=0.2, T_step=0.1,\n        thermalization=1000, measurement=2000, measure_every=10,\n        replicas=32,\n        config_every=1,                   # save every replica\'s configuration every this many T steps\n        seed=1,\n    )\n\n\ndef _kB(units: str) -> float:\n    u = str(units).strip().lower()\n    if u not in ("reduced", "mev"):\n        raise ValueError("units must be \'reduced\' (kB = 1) or \'meV\' (T in kelvin)")\n    return 1.0 if u == "reduced" else KB_MEV\n\n\ndef prepare(cfg: dict, out, log=print) -> dict:\n    """Validate a configuration, write run.json and sites.npy, and allocate configurations.npy."""\n    out = Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    run = {**default_config(), **cfg}\n    if not run["states"]:\n        raise ValueError("define at least one state")\n    kB = _kB(run["units"])\n    temps = temperature_ladder(run["T_max"], run["T_min"], run["T_step"])\n    pos, mask = active_sites(run["crystal"], run["box"], run["coords"], run["shape"])\n    s0 = build_system(run["model"], run["crystal"], run["box"], run["periodic"], run["states"][0], mask, log)\n    ce = max(1, int(run["config_every"]))\n    n_saved = len(range(0, len(temps), ce))\n    n_sites = int(mask.sum())\n    run.update(kB=kB, temperatures=temps.tolist(), periodic_used=list(s0.periodic), n_sites=n_sites,\n               n_saved=n_saved, config_temperatures=temps[::ce].tolist())\n    (out / "run.json").write_text(json.dumps(run, indent=1))\n    np.save(out / "sites.npy", pos[mask].astype(np.float32))\n    np.save(out / "mask.npy", mask)\n    shape = (len(run["states"]), int(run["replicas"]), n_saved, n_sites, 3)\n    np.lib.format.open_memmap(out / "configurations.npy", mode="w+", dtype=np.float32, shape=shape).flush()\n    gb = size_estimate_gb(*shape[:4])\n    log(f"{len(run[\'states\'])} state(s) x {run[\'replicas\']} replicas x {len(temps)} temperatures, "\n        f"{n_sites} active sites; configurations.npy = {gb:.3f} GB")\n    return run\n\n\ndef run(cfg: dict, state_indices, out, device="cpu", log=print) -> None:\n    """Simulate the given states of a prepared run and write state_XXX.npz and their configurations."""\n    out = Path(out)\n    mask = np.load(out / "mask.npy")\n    cf = np.load(out / "configurations.npy", mmap_mode="r+")\n    for i in state_indices:\n        theta = cfg["states"][i]\n        log(f"state {i} ({theta.get(\'name\', i)}) on {device}")\n        s = build_system(cfg["model"], cfg["crystal"], cfg["box"], cfg["periodic_used"], theta, mask, log)\n        r = simulate_state(s, cfg["replicas"], cfg["temperatures"], int(cfg["thermalization"]),\n                           int(cfg["measurement"]), int(cfg["measure_every"]), int(cfg["config_every"]),\n                           int(cfg["seed"]) + 1000 * int(i), device, float(cfg["kB"]), log)\n        cf[i] = r.pop("configs")\n        cf.flush()\n        np.savez(out / f"state_{i:03d}.npz", **r)\n    del cf\n\n\ndef collect(out) -> dict:\n    """Combine the state files: arrays (n_states, replicas, nT[, 3]) and Cv, chi per site (n_states, nT)."""\n    out = Path(out)\n    cfg = json.loads((out / "run.json").read_text())\n    T = np.asarray(cfg["temperatures"])\n    N, kB = float(cfg["n_sites"]), float(cfg["kB"])\n    parts = [np.load(out / f"state_{i:03d}.npz") for i in range(len(cfg["states"]))]\n    res = {k: np.stack([p[k] for p in parts]) for k in ("M", "E", "E2", "Mabs", "M2")}\n    varE = np.maximum(res["E2"] - res["E"] ** 2, 0.0)\n    varM = np.maximum(res["M2"] - res["Mabs"] ** 2, 0.0)\n    combined = dict(temperatures=T, magnetization=res["M"], energy=res["E"], energy2=res["E2"],\n                    Mabs=res["Mabs"], M2=res["M2"],\n                    Cv=(varE / (kB * T ** 2 * N)).mean(axis=1), chi=(N * varM / (kB * T)).mean(axis=1),\n                    E_site=(res["E"] / N).mean(axis=1), M_site=res["Mabs"].mean(axis=1),\n                    state_names=np.array([str(s.get("name", i)) for i, s in enumerate(cfg["states"])]))\n    np.save(out / "magnetization.npy", combined["magnetization"])\n    np.savez(out / "results.npz", **combined)\n    return combined\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("--run", required=True, help="run directory holding run.json")\n    ap.add_argument("--states", required=True, help="comma-separated state indices")\n    ap.add_argument("--device", default="cuda")\n    a = ap.parse_args(argv)\n    cfg = json.loads((Path(a.run) / "run.json").read_text())\n    run(cfg, [int(x) for x in a.states.split(",") if x != ""], a.run, a.device, lambda m: print(m, flush=True))\n\n\nif __name__ == "__main__":\n    main()\n'
assert hashlib.sha256(_src.encode()).hexdigest() == "d97c96ad76f4aa4d9b6cf261222da69996f9683e4f18e77a42bf12cc4e2fed8a", "simulate.py: sha256 mismatch"
os.makedirs("Spin_Colour_Kernel", exist_ok=True)
with open("Spin_Colour_Kernel/simulate.py", "w") as f:
    f.write(_src)
print("wrote Spin_Colour_Kernel/simulate.py  sha256 d97c96ad76f4aa4d…")

## 4 · Reading and checking the parameters

In [ ]:
# 4 · Read and check the parameters, build the first state, and show the size of the output.
import json
import numpy as np
sys.path.insert(0, os.getcwd())
import Spin_Colour_Kernel.simulate as SIM

cfg = dict(model=MODEL, crystal=CRYSTAL, box=list(BOX), periodic=list(PERIODIC), coords=COORDS, shape=SHAPE,
           states=[dict(s) for s in STATES], units=UNITS, T_max=T_MAX, T_min=T_MIN, T_step=T_STEP,
           thermalization=THERMALIZATION, measurement=MEASUREMENT, measure_every=MEASURE_EVERY,
           replicas=REPLICAS, config_every=CONFIG_EVERY, seed=SEED)
for i, s in enumerate(cfg["states"]):
    s.setdefault("name", f"state-{i + 1}")
    s["J"] = {str(k): float(v) for k, v in (s.get("J") or {1: 1.0}).items()}
names = [s["name"] for s in cfg["states"]]
if len(set(names)) != len(names):
    raise SystemExit("state names must be distinct")
RUN = SIM.prepare(cfg, OUTPUT_DIR)
N_STATES = len(names)
s0 = SIM.build_system(MODEL, CRYSTAL, BOX, RUN["periodic_used"], RUN["states"][0], np.load(f"{OUTPUT_DIR}/mask.npy"))
print(f"colouring: {s0.colouring.n_colours} colours, checked admissible on the box; "
      f"periodic {RUN['periodic_used']}")
print(f"temperatures: {len(RUN['temperatures'])} from {RUN['temperatures'][0]} to {RUN['temperatures'][-1]}")
print(f"configurations saved at {RUN['n_saved']} temperature(s) per state "
      "(the run goes on whatever the size; make sure the disk can hold it)")

## 5 · The run

In [ ]:
# 5 · The run: one process per GPU with the states dealt round-robin, or one process on the CPU.
import threading, time

t0 = time.perf_counter()
if DEVICE == "cpu":
    SIM.run(RUN, list(range(N_STATES)), OUTPUT_DIR, device="cpu", log=print)
else:
    devs = usable[:N_STATES]
    procs = []
    for j, g in enumerate(devs):
        mine = [i for i in range(N_STATES) if i % len(devs) == j]
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONPATH=os.getcwd())
        cmd = [sys.executable, "-u", "-m", "Spin_Colour_Kernel.simulate", "--run", OUTPUT_DIR,
               "--states", ",".join(map(str, mine)), "--device", "cuda:0"]
        procs.append((g, subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                          text=True)))
        print(f"GPU {g}: states {[names[i] for i in mine]}")

    def _pump(g, p):
        for line in p.stdout:
            print(f"GPU {g} {line}", end="", flush=True)

    th = [threading.Thread(target=_pump, args=gp) for gp in procs]
    for t in th: t.start()
    for t in th: t.join()
    bad = [g for g, p in procs if p.wait() != 0]
    if bad:
        raise SystemExit(f"failed on GPU(s) {bad}: read the lines above")
print(f"\ntotal: {(time.perf_counter() - t0) / 60:.2f} min")

## 6 · Results and files

In [ ]:
# 6 · The results, combined over states, and what each file holds.
res = SIM.collect(OUTPUT_DIR)
T = res["temperatures"]
print(f"{'state':<16}{'T':>9}{'E/site':>11}{'|M|':>9}{'Cv':>10}{'χ':>10}")
for i, nm in enumerate(names):
    print(f"{nm:<16}{T[-1]:>9.3f}{res['E_site'][i, -1]:>11.5f}{res['M_site'][i, -1]:>9.4f}"
          f"{res['Cv'][i, -1]:>10.4f}{res['chi'][i, -1]:>10.4f}")
print(f"""
{OUTPUT_DIR}/
  magnetization.npy    {res['magnetization'].shape}  [state, replica, temperature, component]  <M> per site
  configurations.npy   {tuple(np.load(f"{OUTPUT_DIR}/configurations.npy", mmap_mode="r").shape)}  [state, replica, saved T, site, component]
  sites.npy            positions of the active sites, the 'site' axis above
  results.npz          temperatures, energy and energy2 (<E>, <E²> per replica, totals), Mabs (<|M|>), M2 (<|M|²>),
                       Cv = (<E²>−<E>²)/(k_B T² N) and χ = N(<|M|²>−<|M|>²)/(k_B T), replica means [state, T]
  run.json             every setting of this run, the ladder and the saved temperatures
""")

## 7 · Figures

In [ ]:
# 7 · Energy, magnetization, specific heat and susceptibility against temperature.
import matplotlib.pyplot as plt

unit = "K" if UNITS.lower() == "mev" else "J/k_B"
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for i, nm in enumerate(names):
    for a, k, t in ((ax[0, 0], "E_site", "energy per site"), (ax[0, 1], "M_site", "|M| per site"),
                    (ax[1, 0], "Cv", "Cv per site"), (ax[1, 1], "chi", "χ")):
        a.plot(T, res[k][i], "o-", ms=3, lw=1.2, label=nm)
        a.set_xlabel(f"T ({unit})"); a.set_ylabel(t); a.grid(alpha=.25)
ax[0, 0].legend(fontsize=9)
fig.suptitle(f"{MODEL} on {CRYSTAL} {list(BOX)} · {N_STATES} state(s) · {REPLICAS} replicas", y=1.0)
plt.tight_layout(); plt.show()